Track 1 code: without static features

In [ ]:
import os
import numpy as np
import pandas as pd
import time
import json
import warnings
import tempfile
import joblib
import pickle
warnings.filterwarnings('ignore')


# Model and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import torch
import random

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pytorch_tabnet.tab_model import TabNetRegressor

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================================================
# CREATE DIRECTORY FOR SAVING FINAL MODELS
# ======================================================
os.makedirs('final_models_for_shap', exist_ok=True)
os.makedirs('final_models_for_shap/scalers', exist_ok=True)
os.makedirs('final_models_for_shap/feature_names', exist_ok=True)
os.makedirs('final_models_for_shap/test_data', exist_ok=True)

print(" Created directory: final_models_for_shap/")
print("   This directory will contain all final models for SHAP analysis")

# ======================================================
# 1. EXPLICIT FEATURE GROUP DEFINITION (NO INFERENCE!)
# ======================================================

def define_feature_groups_explicit(df):
    """
    STRICT, NAME-BASED feature grouping aligned with methodology.
    Excludes all static features explicitly to ensure correct feature counts.
    """

    # ======================================================
    # 1️⃣ List of columns to exclude (target + identifiers + static features)
    # ======================================================
    exclude_cols = [
        'year', 'yield', 'STATE', 'GEOID', 'X', 'Y', 'Unnamed: 0',  # basic identifiers
        'awc', 'aws', 'b_density', 'cec', 'clay_percent', 'field_capacity',  # soil/topo
        'organic_matter', 'pH', 'saturated_hc', 'sand_percent', 'wilting_point'  # farm/management
    ]
    
    # All remaining columns will be candidate features
    all_features = [c for c in df.columns if c not in exclude_cols]

    # ======================================================
    # 2️⃣ Initialize feature groups
    # ======================================================
    G0, G1, G2, G3, G4, G5 = [], [], [], [], [], []

    # ======================================================
    # 3️⃣ Assign features to groups based on suffix or name patterns
    # ======================================================
    for f in all_features:

        # ---- G5: Spatial anomalies (highest priority)
        if f.endswith('_z'):
            G5.append(f)

        # ---- G4: Efficiency metrics
        elif f in ['NDVI_PPT_efficiency', 'GPP_PPT_efficiency']:
            G4.append(f)

        # ---- G3: Climate stress indicators
        elif f in ['T_range_season_mean', 'Heat_stress_months']:
            G3.append(f)

        # ---- G2: Phenological phase features
        elif any(f.endswith(suffix) for suffix in ['_early_mean', '_peak_mean', '_late_mean']):
            G2.append(f)

        # ---- G1: Seasonal aggregates
        elif any(f.endswith(suffix) for suffix in ['_season_mean', '_season_sum', '_season_std', '_season_max']):
            G1.append(f)

        # ---- G0: Raw monthly + other features
        else:
            G0.append(f)

    # ======================================================
    # 4️ Return sorted feature groups
    # ======================================================
    return {
        'G0': sorted(G0),
        'G1': sorted(G1),
        'G2': sorted(G2),
        'G3': sorted(G3),
        'G4': sorted(G4),
        'G5': sorted(G5)
    }


# ======================================================
# 2. DATA PREPARATION WITH VALIDATION CHECKS
# ======================================================

def prepare_ablation_data(train_df, test_df, experiment_features):
    """
    Prepare data for a specific ablation experiment.
    """
    # Check if all features exist
    missing_train = [f for f in experiment_features if f not in train_df.columns]
    missing_test = [f for f in experiment_features if f not in test_df.columns]
    
    if missing_train or missing_test:
        print(f"  ⚠️ Warning: Some features missing in dataset")
        if missing_train:
            print(f"     Missing in train: {missing_train[:5]}...")
        if missing_test:
            print(f"     Missing in test: {missing_test[:5]}...")
        
        # Use only features that exist
        experiment_features = [f for f in experiment_features 
                             if f in train_df.columns and f in test_df.columns]
        print(f"   Using {len(experiment_features)} available features")
    
    if len(experiment_features) == 0:
        print("   ERROR: No features available for this experiment!")
        return None
    
    # Separate features and target
    X_train = train_df[experiment_features].copy()
    y_train = train_df['yield'].values
    
    X_test = test_df[experiment_features].copy()
    y_test = test_df['yield'].values
    
    # Handle any remaining NaNs
    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_test.mean())
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Convert to PyTorch tensors for deep learning models
    X_train_tensor = torch.FloatTensor(X_train_scaled)
    y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1)
    X_test_tensor = torch.FloatTensor(X_test_scaled)
    y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1)
    
    return {
        'X_train': X_train_scaled, 'y_train': y_train,
        'X_test': X_test_scaled, 'y_test': y_test,
        'X_train_tensor': X_train_tensor, 'y_train_tensor': y_train_tensor,
        'X_test_tensor': X_test_tensor, 'y_test_tensor': y_test_tensor,
        'feature_names': experiment_features,
        'n_features': len(experiment_features),
        'scaler': scaler,
        'X_train_raw': X_train,
        'X_test_raw': X_test
    }

# ======================================================
# 3. COMPREHENSIVE METRICS CALCULATION
# ======================================================

def calculate_comprehensive_metrics(y_true, y_pred, model_name, 
                                  train_time, inference_time=None,
                                  model_size_mb=0, n_features=0, n_params=0):
    """
    Calculate all performance and efficiency metrics for ablation study.
    """
    try:
        # Performance metrics
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        
        # Percentage metrics
        epsilon = 1e-10
        mape = np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
        
        # Advanced statistical metrics
        residuals = y_true - y_pred
        explained_variance = 1 - (np.var(residuals) / (np.var(y_true) + epsilon))
        mbe = np.mean(y_pred - y_true)
        
        # NSE and Index of Agreement
        nse = 1 - (np.sum(residuals ** 2) / (np.sum((y_true - np.mean(y_true)) ** 2) + epsilon))
        
        mean_true = np.mean(y_true)
        denominator = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
        d = 1 - (np.sum(residuals ** 2) / (denominator + epsilon)) if denominator > 0 else 0
        
        # RPD and RPIQ
        rpd = np.std(y_true) / (rmse + epsilon)
        iqr = np.percentile(y_true, 75) - np.percentile(y_true, 25)
        rpiq = iqr / (rmse + epsilon) if rmse > 0 else 0
        
        # Coverage probability
        std_residuals = np.std(residuals) if len(residuals) > 1 else 1.0
        coverage_95 = np.mean((y_pred - 1.96*std_residuals <= y_true) & 
                             (y_true <= y_pred + 1.96*std_residuals))
        
        # Bias statistics
        bias_percent = ((np.mean(y_pred) - np.mean(y_true)) / (np.mean(y_true) + epsilon)) * 100
        
        # Compile all metrics
        metrics = {
            # Experiment Info
            'Model': model_name,
            
            # Performance Metrics
            'MSE': float(mse),
            'RMSE': float(rmse),
            'MAE': float(mae),
            'R²': float(r2),
            'MAPE (%)': float(mape),
            'Explained Variance': float(explained_variance),
            'MBE': float(mbe),
            'NSE': float(nse),
            'Index of Agreement (d)': float(d),
            'RPD': float(rpd),
            'RPIQ': float(rpiq),
            'Coverage 95%': float(coverage_95),
            'Std of Residuals': float(std_residuals),
            'Bias %': float(bias_percent),
            
            # Efficiency Metrics
            'Training Time (s)': float(train_time),
            'Inference Time (s)': float(inference_time) if inference_time else 0.0,
            'Model Size (MB)': float(model_size_mb),
            'Num Features': int(n_features),
            'Num Parameters': int(n_params),
        }
        
        return metrics
        
    except Exception as e:
        print(f"     Metrics calculation error: {e}")
        # Return basic metrics if calculation fails
        return {
            'Model': model_name,
            'RMSE': float(rmse) if 'rmse' in locals() else 0.0,
            'MAE': float(mae) if 'mae' in locals() else 0.0,
            'R²': float(r2) if 'r2' in locals() else 0.0,
            'Training Time (s)': float(train_time),
            'Num Features': int(n_features),
        }

# ======================================================
# 4. OPTIMIZED MODELS FOR ABLATION (FIXED HYPERPARAMETERS)
# ======================================================

class AblationModelTrainer:
    """Train models with fixed hyperparameters for fair ablation comparison"""
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.final_models = {}  # Dictionary to store final models (E5 only)
        self.final_scalers = {}  # Dictionary to store final scalers
        self.final_feature_names = {}  # Dictionary to store final feature names
        self.final_test_data = {}  # Dictionary to store final test data
    
    def save_final_model(self, model, model_name, data_dict, feature_names, scaler):
        """Save ONLY final model (E5) for SHAP analysis"""
        try:
            print(f"\n     SAVING FINAL MODEL FOR SHAP: {model_name}")
            
            # Create directories if they don't exist
            os.makedirs('final_models_for_shap', exist_ok=True)
            
            # Save the model
            if model_name == 'XGBoost':
                model_filename = f'final_models_for_shap/{model_name}_model.pkl'
                joblib.dump(model, model_filename)
                print(f"      Model saved to: {model_filename}")
                
            elif model_name == 'TabNet':
                model_filename = f'final_models_for_shap/{model_name}_model.zip'
                model.save_model(model_filename)
                print(f"      Model saved to: {model_filename}")
            
            # Save the scaler
            scaler_filename = f'final_models_for_shap/{model_name}_scaler.pkl'
            joblib.dump(scaler, scaler_filename)
            print(f"      Scaler saved to: {scaler_filename}")
            
            # Save feature names
            features_filename = f'final_models_for_shap/{model_name}_features.json'
            with open(features_filename, 'w') as f:
                json.dump({
                    'feature_names': feature_names,
                    'n_features': len(feature_names),
                    'model_name': model_name,
                    'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
                }, f, indent=4)
            print(f"      Feature names saved to: {features_filename}")
            
            # Save test data for SHAP (sampled to avoid memory issues)
            test_data_filename = f'final_models_for_shap/{model_name}_test_data.pkl'
            
            # Sample 500 instances for SHAP (or use all if less than 500)
            n_samples = min(500, len(data_dict['X_test_raw']))
            indices = np.random.choice(len(data_dict['X_test_raw']), n_samples, replace=False)
            
            test_data_sample = {
                'X_test': data_dict['X_test_raw'].iloc[indices],
                'y_test': data_dict['y_test'][indices],
                'X_test_scaled': data_dict['X_test'][indices]
            }
            
            joblib.dump(test_data_sample, test_data_filename)
            print(f"      Test data saved to: {test_data_filename}")
            
            # Store in dictionary for later use
            self.final_models[model_name] = model
            self.final_scalers[model_name] = scaler
            self.final_feature_names[model_name] = feature_names
            self.final_test_data[model_name] = test_data_sample
            
            print(f"    {model_name} saved successfully for SHAP analysis!")
            
            return True
            
        except Exception as e:
            print(f"     Error saving {model_name} model: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def train_xgboost(self, X_train, y_train, X_test, y_test, n_features, 
                     experiment_name, data_dict, feature_names, scaler):
        """Train XGBoost with fixed hyperparameters"""
        start_time = time.time()
        
        try:
            # Fixed hyperparameters for fair comparison
            params = {
                'n_estimators': 300,
                'learning_rate': 0.05,
                'max_depth': 6,
                'min_child_weight': 1,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'reg_alpha': 0.1,
                'reg_lambda': 1.0,
                'random_state': self.random_state,
                'n_jobs': -1,
                'verbosity': 0
            }
            
            model = xgb.XGBRegressor(**params)
            model.fit(X_train, y_train)
            
            # Measure inference time
            inf_start = time.time()
            y_pred = model.predict(X_test)
            inference_time = time.time() - inf_start
            
            train_time = time.time() - start_time
            
            # Calculate model size
            model_size = 0.0
            try:
                with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as tmp:
                    tmp_path = tmp.name
                joblib.dump(model, tmp_path)
                model_size = os.path.getsize(tmp_path) / (1024 * 1024)
                os.unlink(tmp_path)
            except:
                model_size = 0.0
            
            # Count parameters (approximate for tree ensembles)
            n_trees = params['n_estimators']
            avg_leaves = 2 ** (params['max_depth'] - 1)
            n_params_approx = n_trees * avg_leaves
            
            # Save model ONLY if it's the final experiment (E5)
            if experiment_name == 'E5':
                self.save_final_model(model, 'XGBoost', data_dict, feature_names, scaler)
            
            return y_pred, train_time, inference_time, model_size, n_params_approx, model
            
        except Exception as e:
            print(f"     XGBoost training error: {e}")
            dummy_pred = np.zeros_like(y_test)
            return dummy_pred, time.time()-start_time, 0.0, 0.0, 0, None
    
    def train_tabnet(self, X_train, y_train, X_test, y_test, n_features,
                    experiment_name, data_dict, feature_names, scaler):
        """Train TabNet with fixed hyperparameters"""
        start_time = time.time()
        
        try:
            # Fixed hyperparameters for fair comparison
            model = TabNetRegressor(
                n_d=16,
                n_a=16,
                n_steps=3,
                gamma=1.3,
                lambda_sparse=1e-3,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=2e-2),
                mask_type='entmax',
                scheduler_params={
                    "mode": "min",
                    "patience": 10,
                    "min_lr": 1e-5,
                    "factor": 0.5,
                },
                scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
                verbose=0,
                seed=self.random_state
            )
            
            model.fit(
                X_train, y_train.reshape(-1, 1),
                eval_set=[(X_test, y_test.reshape(-1, 1))],
                max_epochs=100,
                patience=15,
                batch_size=256,
                virtual_batch_size=64,
                eval_metric=['rmse']
            )
            
            inf_start = time.time()
            y_pred = model.predict(X_test).flatten()
            inference_time = time.time() - inf_start
            
            train_time = time.time() - start_time
            
            # Model size
            model_size = 0.0
            try:
                with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp:
                    tmp_path = tmp.name
                model.save_model(tmp_path)
                model_size = os.path.getsize(tmp_path) / (1024 * 1024)
                os.unlink(tmp_path)
            except:
                model_size = 0.0
            
            # Approximate parameter count for TabNet
            n_params_approx = (n_features * 32 * 2 +
                             32 * 32 * 3 * 2 +
                             32 * 1)
            
            # Save model ONLY if it's the final experiment (E5)
            if experiment_name == 'E5':
                self.save_final_model(model, 'TabNet', data_dict, feature_names, scaler)
            
            return y_pred, train_time, inference_time, model_size, n_params_approx, model
            
        except Exception as e:
            print(f"     TabNet training error: {e}")
            dummy_pred = np.zeros_like(y_test)
            return dummy_pred, time.time()-start_time, 0.0, 0.0, 0, None

# ======================================================
# 5. STRICT VALIDATION FUNCTIONS
# ======================================================

def validate_feature_groups(feature_groups, train_df, test_df):
    """
    Perform mandatory validation checks on feature groups.
    """
    print("\n" + "="*60)
    print("🔍 VALIDATING FEATURE GROUPS")
    print("="*60)
    
    all_features = []
    validation_passed = True
    
    # 1. Check feature exclusivity (no overlap between groups)
    print("\n Checking feature exclusivity...")
    for group_name, features in feature_groups.items():
        for other_group, other_features in feature_groups.items():
            if group_name != other_group:
                overlap = set(features) & set(other_features)
                if overlap:
                    print(f"   ❌ OVERLAP DETECTED between {group_name} and {other_group}:")
                    print(f"      Overlapping features: {list(overlap)[:5]}...")
                    validation_passed = False
        all_features.extend(features)
    
    # 2. Check existence in datasets
    print("\n Checking feature existence in datasets...")
    missing_in_train = []
    missing_in_test = []
    
    for feature in all_features:
        if feature not in train_df.columns:
            missing_in_train.append(feature)
        if feature not in test_df.columns:
            missing_in_test.append(feature)
    
    if missing_in_train:
        print(f"     {len(missing_in_train)} features missing in train set (will be filtered out)")
    else:
        print("    All features exist in train set")
    
    if missing_in_test:
        print(f"     {len(missing_in_test)} features missing in test set (will be filtered out)")
    else:
        print("    All features exist in test set")
    
    # 3. Check for duplicates in all_features
    print("\n Checking for duplicate features...")
    feature_counts = {}
    for feature in all_features:
        feature_counts[feature] = feature_counts.get(feature, 0) + 1
    
    duplicates = [feat for feat, count in feature_counts.items() if count > 1]
    if duplicates:
        print(f"   {len(duplicates)} duplicate features found:")
        print(f"      Sample duplicates: {duplicates[:5]}...")
        validation_passed = False
    else:
        print("   No duplicate features found")
    
    # 4. Print summary statistics
    print("\n Feature group statistics:")
    for group_name, features in feature_groups.items():
        existing_features = [f for f in features if f in train_df.columns and f in test_df.columns]
        print(f"   {group_name}: {len(existing_features)}/{len(features)} features (existing/total)")
    
    print(f"\n   Total unique features: {len(set(all_features))}")
    
    if validation_passed:
        print("\n ALL VALIDATION CHECKS PASSED!")
    else:
        print("\n  Validation warnings detected (will attempt to continue)")
    
    print("="*60)
    
    return validation_passed, set(all_features)

def validate_experiment_progression(experiments):
    """
    Validate that experiments follow monotonic progression: E0 ⊂ E1 ⊂ ... ⊂ E5
    """
    print("\n" + "="*60)
    print("🔍 VALIDATING EXPERIMENT PROGRESSION")
    print("="*60)
    
    validation_passed = True
    prev_experiment = None
    prev_features = set()
    
    for exp_name in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']:
        if exp_name not in experiments:
            print(f"    Missing experiment: {exp_name}")
            validation_passed = False
            continue
        
        current_features = set(experiments[exp_name])
        
        # Check if experiment is subset of next experiment
        if prev_experiment:
            if not prev_features.issubset(current_features):
                missing = prev_features - current_features
                print(f"    {prev_experiment} is NOT a subset of {exp_name}")
                print(f"      Missing features: {list(missing)[:5]}...")
        
        # Check monotonic increase in feature count
        if prev_experiment:
            if len(current_features) <= len(prev_features):
                print(f"    Feature count not increasing: {exp_name} ({len(current_features)}) <= {prev_experiment} ({len(prev_features)})")
            else:
                print(f"    Feature count increased: {len(prev_features)} → {len(current_features)} (+{len(current_features) - len(prev_features)})")
        
        prev_experiment = exp_name
        prev_features = current_features
    
    print("\n EXPERIMENT PROGRESSION:")
    print(f"   E0 ({len(set(experiments['E0']))})")
    print(f"   E1 ({len(set(experiments['E1']))})")
    print(f"   E2 ({len(set(experiments['E2']))})")
    print(f"   E3 ({len(set(experiments['E3']))})")
    print(f"   E4 ({len(set(experiments['E4']))})")
    print(f"   E5 ({len(set(experiments['E5']))})")
    
    print("="*60)
    
    return True  # Always continue even with warnings

# ======================================================
# 6. ABLATION EXPERIMENT PIPELINE (CORRECTED)
# ======================================================

def run_ablation_experiments(train_df, test_df, 
                           models_to_use=['XGBoost', 'TabNet'], 
                           output_dir='ablation_results'):
    """
    Main function to run all ablation experiments with strict validation.
    """
    print("="*80)
    print("STARTING COMPREHENSIVE ABLATION STUDY")
    print("="*80)
    print(" Using automatic feature detection from dataset")
    print(" Final models (E5 only) will be saved for SHAP analysis")
    print("="*80)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # ======================================================
    # STEP 1: Define feature groups automatically from data
    # ======================================================
    print("\n DETECTING FEATURE GROUPS FROM DATASET...")
    feature_groups = define_feature_groups_explicit(train_df)
    
    # ======================================================
    # STEP 2: Validate feature groups
    # ======================================================
    validation_passed, all_unique_features = validate_feature_groups(feature_groups, train_df, test_df)
    
    # ======================================================
    # STEP 3: Define experiments with cumulative progression
    # ======================================================
    print("\n DEFINING ABLATION EXPERIMENTS...")
    
    # Filter features to only those that exist in both datasets
    for group_name in feature_groups:
        feature_groups[group_name] = [f for f in feature_groups[group_name] 
                                     if f in train_df.columns and f in test_df.columns]
    
    # E0 = G0
    E0 = feature_groups['G0']
    
    # E1 = G0 + G1
    E1 = feature_groups['G0'] + feature_groups['G1']
    
    # E2 = G0 + G1 + G2
    E2 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2']
    
    # E3 = G0 + G1 + G2 + G3
    E3 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2'] + feature_groups['G3']
    
    # E4 = G0 + G1 + G2 + G3 + G4
    E4 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2'] + feature_groups['G3'] + feature_groups['G4']
    
    # E5 = G0 + G1 + G2 + G3 + G4 + G5
    E5 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2'] + feature_groups['G3'] + feature_groups['G4'] + feature_groups['G5']
    
    # Remove duplicates while preserving order
    experiments = {
        'E0': list(dict.fromkeys(E0)),
        'E1': list(dict.fromkeys(E1)),
        'E2': list(dict.fromkeys(E2)),
        'E3': list(dict.fromkeys(E3)),
        'E4': list(dict.fromkeys(E4)),
        'E5': list(dict.fromkeys(E5))
    }
    
    # ======================================================
    # STEP 4: Validate experiment progression
    # ======================================================
    progression_passed = validate_experiment_progression(experiments)
    
    # ======================================================
    # STEP 5: Print experiment summary
    # ======================================================
    print("\n EXPERIMENT SUMMARY:")
    exp_display_names = {}
    for exp_name in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']:
        features = experiments[exp_name]
        
        # Create display name
        if exp_name == 'E0':
            display_name = 'E0 (G0 only)'
        elif exp_name == 'E1':
            display_name = 'E1 (G0+G1)'
        elif exp_name == 'E2':
            display_name = 'E2 (G0+G1+G2)'
        elif exp_name == 'E3':
            display_name = 'E3 (G0+G1+G2+G3)'
        elif exp_name == 'E4':
            display_name = 'E4 (G0+G1+G2+G3+G4)'
        elif exp_name == 'E5':
            display_name = 'E5 (All features)'
        
        exp_display_names[exp_name] = display_name
        
        print(f"  {display_name}")
        print(f"    Features: {len(features)}")
        if features:
            print(f"    Sample features: {features[:3]}...")
        else:
            print(f"      No features available!")
    
    # ======================================================
    # STEP 6: Run experiments
    # ======================================================
    print(f"\n MODELS TO TRAIN: {models_to_use}")
    print(" Only E5 models will be saved for SHAP analysis")
    
    # Initialize results storage
    all_results = []
    trainer = AblationModelTrainer()
    
    # Run each experiment
    for exp_name in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']:
        print(f"\n{'='*60}")
        print(f"🏃‍♂️ RUNNING EXPERIMENT: {exp_display_names[exp_name]}")
        print(f"{'='*60}")
        
        # Get features for this experiment
        exp_features = experiments[exp_name]
        
        if len(exp_features) == 0:
            print(f"    No features available for this experiment, skipping...")
            continue
        
        # Prepare data for this experiment
        data = prepare_ablation_data(train_df, test_df, exp_features)
        
        if data is None:
            print(f"   Failed to prepare data for experiment {exp_name}")
            continue
        
        print(f"  Features: {len(exp_features)}")
        print(f"  Train samples: {len(data['y_train'])}")
        print(f"  Test samples: {len(data['y_test'])}")
        
        # Train each model
        for model_name in models_to_use:
            print(f"\n  Training {model_name}...")
            
            try:
                if model_name == 'XGBoost':
                    y_pred, train_time, inf_time, model_size, n_params, model = \
                        trainer.train_xgboost(data['X_train'], data['y_train'],
                                            data['X_test'], data['y_test'],
                                            data['n_features'],
                                            exp_name, data,
                                            data['feature_names'], data['scaler'])
                
                elif model_name == 'TabNet':
                    y_pred, train_time, inf_time, model_size, n_params, model = \
                        trainer.train_tabnet(data['X_train'], data['y_train'],
                                           data['X_test'], data['y_test'],
                                           data['n_features'],
                                           exp_name, data,
                                           data['feature_names'], data['scaler'])
                
                else:
                    print(f"     Unknown model: {model_name}")
                    continue
                
                # Check if we got valid predictions
                if y_pred is None or len(y_pred) == 0:
                    print(f"     No predictions returned for {model_name}")
                    continue
                
                # Calculate metrics
                metrics = calculate_comprehensive_metrics(
                    data['y_test'], y_pred, model_name, train_time, inf_time, 
                    model_size, len(exp_features), n_params
                )
                
                # Add experiment info
                metrics['Experiment'] = exp_display_names[exp_name]
                metrics['Experiment_Code'] = exp_name
                
                all_results.append(metrics)
                
                print(f"     RMSE: {metrics['RMSE']:.4f}, R²: {metrics['R²']:.4f}")
                print(f"      Train: {train_time:.1f}s, Infer: {inf_time:.3f}s")
                print(f"     Features: {len(exp_features)}")
                
                # Special message for E5 experiment
                if exp_name == 'E5':
                    print(f"     Model will be saved for SHAP analysis!")
                
            except Exception as e:
                print(f"    Failed: {str(e)}")
                import traceback
                traceback.print_exc()
    
    # Check if we have results
    if not all_results:
        print("\n WARNING: No results collected!")
        return pd.DataFrame(), experiments, exp_display_names
    
    # ======================================================
    # STEP 7: Create and save results
    # ======================================================
    results_df = pd.DataFrame(all_results)
    
    # Reorder columns for better readability
    available_columns = results_df.columns.tolist()
    column_order = ['Experiment', 'Experiment_Code', 'Model', 'Num Features', 
                   'RMSE', 'MAE', 'R²', 'MAPE (%)', 'NSE', 'Index of Agreement (d)', 
                   'RPD', 'Training Time (s)', 'Inference Time (s)', 
                   'Model Size (MB)', 'Num Parameters', 'MSE', 'Explained Variance',
                   'MBE', 'Std of Residuals', 'Coverage 95%', 'Bias %', 'RPIQ']
    
    # Keep only columns that exist
    column_order = [col for col in column_order if col in available_columns]
    # Add any remaining columns
    remaining_cols = [col for col in available_columns if col not in column_order]
    column_order.extend(remaining_cols)
    
    results_df = results_df[column_order]
    
    # Save results
    results_path = os.path.join(output_dir, 'ablation_results.csv')
    results_df.to_csv(results_path, index=False, float_format='%.6f')
    
    # Save experiment configuration
    config_path = os.path.join(output_dir, 'experiment_configuration.json')
    with open(config_path, 'w') as f:
        config_data = {
            'feature_groups': {k: len(v) for k, v in feature_groups.items()},
            'experiments': {k: len(v) for k, v in experiments.items()},
            'total_unique_features': len(all_unique_features),
            'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
        }
        json.dump(config_data, f, indent=4)
    
    print(f"\n{'='*80}")
    print(f" ABLATION STUDY COMPLETED")
    print(f" Results saved to: {results_path}")
    print(f" Configuration saved to: {config_path}")
    print(f" FINAL MODELS SAVED IN: final_models_for_shap/")
    print(f" Total results: {len(results_df)} rows")
    print(f"{'='*80}")
    
    # Print final summary table
    print("\n FINAL RESULTS SUMMARY:")
    summary_df = results_df[['Experiment', 'Model', 'Num Features', 'RMSE', 'R²']].copy()
    print(summary_df.to_string(index=False))
    
    return results_df, experiments, exp_display_names

# ======================================================
# 7. COMPLETE MODEL LOADING FUNCTION
# ======================================================

def load_final_models_for_shap():
    """
    Load saved final models for SHAP analysis.
    Returns a dictionary with models, scalers, feature names, and test data.
    """
    print("\n" + "="*80)
    print(" LOADING FINAL MODELS FOR SHAP ANALYSIS")
    print("="*80)
    
    base_dir = 'final_models_for_shap'
    
    if not os.path.exists(base_dir):
        print(f" Directory not found: {base_dir}")
        print("   Please run the ablation study first to train and save models.")
        return None
    
    models_dict = {}
    
    # List all model files
    model_files = [f for f in os.listdir(base_dir) if f.endswith(('_model.pkl', '_model.zip'))]
    
    if not model_files:
        print(" No model files found in final_models_for_shap/")
        return None
    
    print(f"\n Found {len(model_files)} model files:")
    
    for model_file in model_files:
        if model_file.endswith('_model.pkl'):
            model_name = model_file.replace('_model.pkl', '')
        elif model_file.endswith('_model.zip'):
            model_name = model_file.replace('_model.zip', '')
        else:
            continue
        
        print(f"\n Loading {model_name}...")
        
        try:
            # Load the model
            if model_name == 'XGBoost':
                model_path = os.path.join(base_dir, f'{model_name}_model.pkl')
                model = joblib.load(model_path)
                print(f"    Model loaded from: {model_path}")
                
            elif model_name == 'TabNet':
                model_path = os.path.join(base_dir, f'{model_name}_model.zip')
                model = TabNetRegressor()
                model.load_model(model_path)
                print(f"    Model loaded from: {model_path}")
            
            # Load scaler
            scaler_path = os.path.join(base_dir, f'{model_name}_scaler.pkl')
            scaler = joblib.load(scaler_path)
            print(f"    Scaler loaded from: {scaler_path}")
            
            # Load feature names
            features_path = os.path.join(base_dir, f'{model_name}_features.json')
            with open(features_path, 'r') as f:
                features_info = json.load(f)
            
            feature_names = features_info['feature_names']
            print(f"    Feature names loaded: {len(feature_names)} features")
            
            # Load test data
            test_data_path = os.path.join(base_dir, f'{model_name}_test_data.pkl')
            test_data = joblib.load(test_data_path)
            print(f"    Test data loaded: {len(test_data['X_test'])} samples")
            
            # Store everything in dictionary
            models_dict[model_name] = {
                'model': model,
                'scaler': scaler,
                'feature_names': feature_names,
                'test_data': test_data,
                'X_test_raw': test_data['X_test'],
                'X_test_scaled': test_data['X_test_scaled'],
                'y_test': test_data['y_test']
            }
            
            print(f"   ✨ {model_name} ready for SHAP analysis!")
            
        except Exception as e:
            print(f"    Error loading {model_name}: {e}")
    
    print(f"\n" + "="*80)
    print(f" LOADED {len(models_dict)} MODELS FOR SHAP ANALYSIS")
    print("="*80)
    
    # Print summary
    for model_name, model_info in models_dict.items():
        print(f"\n {model_name}:")
        print(f"   Features: {len(model_info['feature_names'])}")
        print(f"   Test samples: {len(model_info['y_test'])}")
        print(f"   Sample features: {model_info['feature_names'][:3]}...")
    
    return models_dict

# ======================================================
# 8. RUN SHAP ANALYSIS ON FINAL MODELS
# ======================================================

def run_shap_analysis_on_final_models():
    """
    Run SHAP analysis on the final saved models.
    """
    print("\n" + "="*80)
    print("🔍 RUNNING SHAP ANALYSIS ON FINAL MODELS")
    print("="*80)
    
    # Load models
    models_dict = load_final_models_for_shap()
    
    if not models_dict:
        print(" No models loaded. Cannot run SHAP analysis.")
        return
    
    # Check if SHAP is installed
    try:
        import shap
        print(" SHAP library is available")
    except ImportError:
        print(" SHAP not installed. Please install it first:")
        print("   pip install shap")
        return
    
    # Create directory for SHAP outputs
    shap_output_dir = 'final_models_for_shap/shap_results'
    os.makedirs(shap_output_dir, exist_ok=True)
    
    # Run SHAP for each model
    for model_name, model_info in models_dict.items():
        print(f"\n{'='*60}")
        print(f" Running SHAP analysis for {model_name}")
        print(f"{'='*60}")
        
        model = model_info['model']
        feature_names = model_info['feature_names']
        X_test_scaled = model_info['X_test_scaled']
        
        print(f"  Features: {len(feature_names)}")
        print(f"  Test samples: {X_test_scaled.shape[0]}")
        
        # Sample data for faster SHAP computation
        sample_size = min(100, X_test_scaled.shape[0])
        X_sample = X_test_scaled[:sample_size]
        
        print(f"  Using {sample_size} samples for SHAP computation")
        
        try:
            if model_name == 'XGBoost':
                # Tree-based models use TreeExplainer
                print(f"  Using TreeExplainer for {model_name}")
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_sample)
                
                # Summary plot
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_values, X_sample, 
                                 feature_names=feature_names, 
                                 show=False, plot_size=None)
                plt.title(f'SHAP Summary Plot - {model_name}', fontsize=16)
                plt.tight_layout()
                summary_plot_path = os.path.join(shap_output_dir, f'shap_summary_{model_name}.png')
                plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"   Summary plot saved: {summary_plot_path}")
                
                # Bar plot (mean absolute SHAP values)
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_values, X_sample, 
                                 feature_names=feature_names, 
                                 plot_type="bar", show=False)
                plt.title(f'SHAP Feature Importance - {model_name}', fontsize=16)
                plt.tight_layout()
                bar_plot_path = os.path.join(shap_output_dir, f'shap_bar_{model_name}.png')
                plt.savefig(bar_plot_path, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"   Bar plot saved: {bar_plot_path}")
                
                # Calculate and save feature importance
                shap_values_abs = np.abs(shap_values).mean(0)
                feature_importance = pd.DataFrame({
                    'feature': feature_names,
                    'importance': shap_values_abs
                }).sort_values('importance', ascending=False)
                
                importance_path = os.path.join(shap_output_dir, f'feature_importance_{model_name}.csv')
                feature_importance.to_csv(importance_path, index=False)
                print(f"   Feature importance saved: {importance_path}")
                
                # Top 10 features
                print(f"\n  🏆 Top 10 important features for {model_name}:")
                for i, row in feature_importance.head(10).iterrows():
                    print(f"     {i+1:2d}. {row['feature']}: {row['importance']:.6f}")
                
            elif model_name == 'TabNet':
                # Neural network models use KernelExplainer
                print(f"  Using KernelExplainer for {model_name}")
                
                # Define prediction function
                def predict_fn(X):
                    return model.predict(X).flatten()
                
                # Use a small background dataset
                background = shap.sample(X_sample, 10)
                explainer = shap.KernelExplainer(predict_fn, background)
                
                # Calculate SHAP values for a small sample
                shap_values = explainer.shap_values(X_sample[:50], nsamples=100)
                
                # Summary plot
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_values, X_sample[:50], 
                                 feature_names=feature_names, 
                                 show=False, plot_size=None)
                plt.title(f'SHAP Summary Plot - {model_name}', fontsize=16)
                plt.tight_layout()
                summary_plot_path = os.path.join(shap_output_dir, f'shap_summary_{model_name}.png')
                plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"   Summary plot saved: {summary_plot_path}")
                
                # Calculate and save feature importance
                shap_values_abs = np.abs(shap_values).mean(0)
                feature_importance = pd.DataFrame({
                    'feature': feature_names,
                    'importance': shap_values_abs
                }).sort_values('importance', ascending=False)
                
                importance_path = os.path.join(shap_output_dir, f'feature_importance_{model_name}.csv')
                feature_importance.to_csv(importance_path, index=False)
                print(f"   Feature importance saved: {importance_path}")
                
                # Top 10 features
                print(f"\n   Top 10 important features for {model_name}:")
                for i, row in feature_importance.head(10).iterrows():
                    print(f"     {i+1:2d}. {row['feature']}: {row['importance']:.6f}")
            
            print(f"\n   SHAP analysis complete for {model_name}")
            
        except Exception as e:
            print(f"  Error in SHAP analysis for {model_name}: {e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n" + "="*80)
    print(" SHAP ANALYSIS COMPLETED!")
    print(f" All results saved in: {shap_output_dir}/")
    print("="*80)
    
    # Create a README file for the SHAP results
    readme_content = f"""# SHAP Analysis Results

This directory contains SHAP analysis results for the final models.

## Generated Files:
For each model, you will find:
1. `shap_summary_[model_name].png` - SHAP summary plot
2. `shap_bar_[model_name].png` - SHAP bar plot (feature importance)
3. `feature_importance_[model_name].csv` - Feature importance values

## Models Analyzed:
{', '.join(models_dict.keys())}

## How to Interpret:
1. **Summary Plot**: Shows the impact of each feature on model output
2. **Bar Plot**: Shows mean absolute SHAP values (feature importance)
3. **CSV File**: Contains numerical importance values for each feature

## Date Generated: {time.strftime("%Y-%m-%d %H:%M:%S")}
"""
    
    readme_path = os.path.join(shap_output_dir, 'README.md')
    with open(readme_path, 'w') as f:
        f.write(readme_content)
    
    print(f"\n Documentation saved: {readme_path}")

# ======================================================
# 9. MAIN EXECUTION SCRIPT (WITH BOTH MODELS)
# ======================================================

def main():
    """
    Main execution function for the ablation study with model saving.
    NO INTERACTIVE PROMPTS - RUNS AUTOMATICALLY
    """
    print("="*80)
    print(" CORN YIELD ESTIMATION - ABLATION STUDY (TRACK 1)")
    print("="*80)
    print("Running automatically with NO interactive prompts...")
    print("Training BOTH XGBoost and TabNet models...")
    print("="*80)
    
    # Set your paths here - USE YOUR ACTUAL PATHS
    train_path = "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_train_FE.csv"
    test_path = "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_test_FE.csv"
    
    # Load your feature-engineered data
    print("\n LOADING DATA...")
    try:
        train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)
        
        print(f" Train shape: {train_df.shape}")
        print(f" Test shape: {test_df.shape}")
        
        # Check for required columns
        if 'yield' not in train_df.columns or 'yield' not in test_df.columns:
            print(" ERROR: 'yield' column not found in datasets!")
            print("   Available columns in train:")
            print(f"   {train_df.columns.tolist()[:10]}...")
            return
        
        print(" Data loaded successfully!")
        
    except Exception as e:
        print(f" Error loading data: {e}")
        print("Please check your file paths and ensure the CSV files exist.")
        return
    
    # ======================================================
    #  CRITICAL FIX: BOTH MODELS INCLUDED
    # ======================================================
    models_to_use = ['XGBoost', 'TabNet']
    
    print(f"\n MODELS TO TRAIN: {models_to_use}")
    print(" Only E5 models will be saved for SHAP analysis")
    
    # Run ablation experiments
    try:
        start_time = time.time()
        results_df, experiments, exp_display_names = run_ablation_experiments(
            train_df, test_df, 
            models_to_use=models_to_use,  # This now includes both models
            output_dir='ablation_results'
        )
        
        execution_time = time.time() - start_time
        
        # Final summary
        print("\n" + "="*80)
        print(" ABLATION STUDY COMPLETE (TRACK 1)")
        print("="*80)
        print(f"  Total execution time: {execution_time:.1f} seconds")
        
        # Check if files were saved
        print("\n CHECKING SAVED FILES:")
        
        # Check ablation results
        if os.path.exists('ablation_results/ablation_results.csv'):
            results_size = os.path.getsize('ablation_results/ablation_results.csv') / 1024
            print(f"   ablation_results.csv ({results_size:.1f} KB)")
            
            # Load and display results
            saved_results = pd.read_csv('ablation_results/ablation_results.csv')
            print(f"\n SAVED RESULTS SUMMARY:")
            print(f"  Total experiments: {len(saved_results)}")
            print(f"  Unique models: {saved_results['Model'].unique().tolist()}")
            print(f"  Experiments: {saved_results['Experiment'].unique().tolist()}")
            
            # Show sample of results
            print(f"\n SAMPLE RESULTS (first 3 rows):")
            print(saved_results.head(3).to_string(index=False))
        else:
            print("   ablation_results.csv NOT FOUND!")
        
        # Check saved models
        model_files = []
        if os.path.exists('final_models_for_shap'):
            model_files = os.listdir('final_models_for_shap')
        
        if model_files:
            print(f"\n   final_models_for_shap/ directory contains {len(model_files)} files")
            
            # Count files by model type
            xgb_files = [f for f in model_files if 'XGBoost' in f]
            tabnet_files = [f for f in model_files if 'TabNet' in f]
            print(f"     XGBoost files: {len(xgb_files)}")
            print(f"     TabNet files: {len(tabnet_files)}")
            
            print(f"\n   DETAILED FILE LIST:")
            for file in sorted(model_files):
                if any(x in file for x in ['_model.', '_scaler.', '_features.', '_test_data.']):
                    file_size = os.path.getsize(f'final_models_for_shap/{file}') / 1024
                    print(f"     - {file} ({file_size:.1f} KB)")
        else:
            print("   No files found in final_models_for_shap/")
        
        # Summary message
        print("\n" + "="*80)
        print(" SUCCESS: Track 1 Ablation study completed successfully!")
        print(f" Results saved in: ablation_results/ablation_results.csv")
        print(f" Models saved in: final_models_for_shap/")
        print("\n To run SHAP analysis later, use:")
        print("   run_shap_analysis_on_final_models()")
        print("="*80)
        
    except Exception as e:
        print(f"\n ERROR in ablation study: {e}")
        import traceback
        traceback.print_exc()

# ======================================================
# 10. DIRECT EXECUTION
# ======================================================

if __name__ == "__main__":
    # Create necessary directories
    os.makedirs('ablation_results', exist_ok=True)
    os.makedirs('final_models_for_shap', exist_ok=True)
    
    # Run the ablation study
    print("Starting Track 1 ablation study...")
    print("Training BOTH XGBoost and TabNet models...")
    print("NO interactive prompts - running fully automatically")
    print("-" * 50)
    
    main()
    print("\n Track 1 ablation study execution completed!")

Track 2 code: dynamic plus static features 

In [ ]:
import os
import numpy as np
import pandas as pd
import time
import json
import warnings
import tempfile
import joblib
import pickle
warnings.filterwarnings('ignore')

# Model and evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import torch
import random

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pytorch_tabnet.tab_model import TabNetRegressor

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================================================
# CREATE DIRECTORY FOR SAVING FINAL MODELS
# ======================================================
os.makedirs('final_models_for_shap_T2', exist_ok=True)
os.makedirs('final_models_for_shap_T2/scalers', exist_ok=True)
os.makedirs('final_models_for_shap_T2/feature_names', exist_ok=True)
os.makedirs('final_models_for_shap_T2/test_data', exist_ok=True)

print(" Created directory: final_models_for_shap_T2/")
print("   This directory will contain all final models for SHAP analysis")

# ======================================================
# 1. EXPLICIT FEATURE GROUP DEFINITION (NO INFERENCE!)
# ======================================================

def define_feature_groups_explicit(df):
    """
    STRICT, NAME-BASED feature grouping aligned with methodology.
    NO pattern ambiguity.
    """

    exclude_cols = ['yield', 'GEOID', 'Unnamed: 0']
    all_features = [c for c in df.columns if c not in exclude_cols]

    G0, G1, G2, G3, G4, G5 = [], [], [], [], [], []

    for f in all_features:

        # ---- G5: Spatial anomalies (highest priority)
        if f.endswith('_z'):
            G5.append(f)

        # ---- G4: Efficiency metrics
        elif f in ['NDVI_PPT_efficiency', 'GPP_PPT_efficiency']:
            G4.append(f)

        # ---- G3: Climate stress indicators
        elif f in ['T_range_season_mean', 'Heat_stress_months']:
            G3.append(f)

        # ---- G2: Phenological phase features
        elif any(f.endswith(suffix) for suffix in [
            '_early_mean', '_peak_mean', '_late_mean'
        ]):
            G2.append(f)

        # ---- G1: Seasonal aggregates
        elif any(f.endswith(suffix) for suffix in [
            '_season_mean', '_season_sum', '_season_std', '_season_max'
        ]):
            G1.append(f)

        # ---- G0: Raw monthly + soil + static
        else:
            G0.append(f)

    return {
        'G0': sorted(G0),
        'G1': sorted(G1),
        'G2': sorted(G2),
        'G3': sorted(G3),
        'G4': sorted(G4),
        'G5': sorted(G5)
    }

# ======================================================
# 2. DATA PREPARATION WITH VALIDATION CHECKS
# ======================================================

def prepare_ablation_data(train_df, test_df, experiment_features):
    """
    Prepare data for a specific ablation experiment.
    """
    # Check if all features exist
    missing_train = [f for f in experiment_features if f not in train_df.columns]
    missing_test = [f for f in experiment_features if f not in test_df.columns]
    
    if missing_train or missing_test:
        print(f"   Warning: Some features missing in dataset")
        if missing_train:
            print(f"     Missing in train: {missing_train[:5]}...")
        if missing_test:
            print(f"     Missing in test: {missing_test[:5]}...")
        
        # Use only features that exist
        experiment_features = [f for f in experiment_features 
                             if f in train_df.columns and f in test_df.columns]
        print(f"   Using {len(experiment_features)} available features")
    
    if len(experiment_features) == 0:
        print("   ERROR: No features available for this experiment!")
        return None
    
    # Separate features and target
    X_train = train_df[experiment_features].copy()
    y_train = train_df['yield'].values
    
    X_test = test_df[experiment_features].copy()
    y_test = test_df['yield'].values
    
    # Handle any remaining NaNs
    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_test.mean())
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Convert to PyTorch tensors for deep learning models
    X_train_tensor = torch.FloatTensor(X_train_scaled)
    y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1)
    X_test_tensor = torch.FloatTensor(X_test_scaled)
    y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1)
    
    return {
        'X_train': X_train_scaled, 'y_train': y_train,
        'X_test': X_test_scaled, 'y_test': y_test,
        'X_train_tensor': X_train_tensor, 'y_train_tensor': y_train_tensor,
        'X_test_tensor': X_test_tensor, 'y_test_tensor': y_test_tensor,
        'feature_names': experiment_features,
        'n_features': len(experiment_features),
        'scaler': scaler,
        'X_train_raw': X_train,
        'X_test_raw': X_test
    }

# ======================================================
# 3. COMPREHENSIVE METRICS CALCULATION
# ======================================================

def calculate_comprehensive_metrics(y_true, y_pred, model_name, 
                                  train_time, inference_time=None,
                                  model_size_mb=0, n_features=0, n_params=0):
    """
    Calculate all performance and efficiency metrics for ablation study.
    """
    try:
        # Performance metrics
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        
        # Percentage metrics
        epsilon = 1e-10
        mape = np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
        
        # Advanced statistical metrics
        residuals = y_true - y_pred
        explained_variance = 1 - (np.var(residuals) / (np.var(y_true) + epsilon))
        mbe = np.mean(y_pred - y_true)
        
        # NSE and Index of Agreement
        nse = 1 - (np.sum(residuals ** 2) / (np.sum((y_true - np.mean(y_true)) ** 2) + epsilon))
        
        mean_true = np.mean(y_true)
        denominator = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
        d = 1 - (np.sum(residuals ** 2) / (denominator + epsilon)) if denominator > 0 else 0
        
        # RPD and RPIQ
        rpd = np.std(y_true) / (rmse + epsilon)
        iqr = np.percentile(y_true, 75) - np.percentile(y_true, 25)
        rpiq = iqr / (rmse + epsilon) if rmse > 0 else 0
        
        # Coverage probability
        std_residuals = np.std(residuals) if len(residuals) > 1 else 1.0
        coverage_95 = np.mean((y_pred - 1.96*std_residuals <= y_true) & 
                             (y_true <= y_pred + 1.96*std_residuals))
        
        # Bias statistics
        bias_percent = ((np.mean(y_pred) - np.mean(y_true)) / (np.mean(y_true) + epsilon)) * 100
        
        # Compile all metrics
        metrics = {
            # Experiment Info
            'Model': model_name,
            
            # Performance Metrics
            'MSE': float(mse),
            'RMSE': float(rmse),
            'MAE': float(mae),
            'R²': float(r2),
            'MAPE (%)': float(mape),
            'Explained Variance': float(explained_variance),
            'MBE': float(mbe),
            # 'NSE': float(nse),
            'Index of Agreement (d)': float(d),
            'RPD': float(rpd),
            'RPIQ': float(rpiq),
            'Coverage 95%': float(coverage_95),
            'Std of Residuals': float(std_residuals),
            # 'Bias %': float(bias_percent),
            
            # Efficiency Metrics
            'Training Time (s)': float(train_time),
            'Inference Time (s)': float(inference_time) if inference_time else 0.0,
            # 'Model Size (MB)': float(model_size_mb),
            'Num Features': int(n_features),
            'Num Parameters': int(n_params),
        }
        
        return metrics
        
    except Exception as e:
        print(f"     Metrics calculation error: {e}")
        # Return basic metrics if calculation fails
        return {
            'Model': model_name,
            'RMSE': float(rmse) if 'rmse' in locals() else 0.0,
            'MAE': float(mae) if 'mae' in locals() else 0.0,
            'R²': float(r2) if 'r2' in locals() else 0.0,
            'Training Time (s)': float(train_time),
            'Num Features': int(n_features),
        }

# ======================================================
# 4. OPTIMIZED MODELS FOR ABLATION (FIXED HYPERPARAMETERS)
# ======================================================

class AblationModelTrainer:
    """Train models with fixed hyperparameters for fair ablation comparison"""
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.final_models = {}  # Dictionary to store final models (E5 only)
        self.final_scalers = {}  # Dictionary to store final scalers
        self.final_feature_names = {}  # Dictionary to store final feature names
        self.final_test_data = {}  # Dictionary to store final test data
    
    def save_final_model(self, model, model_name, data_dict, feature_names, scaler):
        """Save ONLY final model (E5) for SHAP analysis"""
        try:
            print(f"\n     SAVING FINAL MODEL FOR SHAP: {model_name}")
            
            # Create directories if they don't exist
            os.makedirs('final_models_for_shap_T2', exist_ok=True)
            
            # Save the model
            if model_name == 'XGBoost':
                model_filename = f'final_models_for_shap_T2/{model_name}_model.pkl'
                joblib.dump(model, model_filename)
                print(f"      Model saved to: {model_filename}")
                
            elif model_name == 'TabNet':
                model_filename = f'final_models_for_shap_T2/{model_name}_model.zip'
                model.save_model(model_filename)
                print(f"      Model saved to: {model_filename}")
            
            # Save the scaler
            scaler_filename = f'final_models_for_shap_T2/{model_name}_scaler.pkl'
            joblib.dump(scaler, scaler_filename)
            print(f"      Scaler saved to: {scaler_filename}")
            
            # Save feature names
            features_filename = f'final_models_for_shap_T2/{model_name}_features.json'
            with open(features_filename, 'w') as f:
                json.dump({
                    'feature_names': feature_names,
                    'n_features': len(feature_names),
                    'model_name': model_name,
                    'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
                }, f, indent=4)
            print(f"      Feature names saved to: {features_filename}")
            
            # Save test data for SHAP (sampled to avoid memory issues)
            test_data_filename = f'final_models_for_shap_T2/{model_name}_test_data.pkl'
            
            # Sample 500 instances for SHAP (or use all if less than 500)
            n_samples = min(500, len(data_dict['X_test_raw']))
            indices = np.random.choice(len(data_dict['X_test_raw']), n_samples, replace=False)
            
            test_data_sample = {
                'X_test': data_dict['X_test_raw'].iloc[indices],
                'y_test': data_dict['y_test'][indices],
                'X_test_scaled': data_dict['X_test'][indices]
            }
            
            joblib.dump(test_data_sample, test_data_filename)
            print(f"      Test data saved to: {test_data_filename}")
            
            # Store in dictionary for later use
            self.final_models[model_name] = model
            self.final_scalers[model_name] = scaler
            self.final_feature_names[model_name] = feature_names
            self.final_test_data[model_name] = test_data_sample
            
            print(f"     {model_name} saved successfully for SHAP analysis!")
            
            return True
            
        except Exception as e:
            print(f"     Error saving {model_name} model: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def train_xgboost(self, X_train, y_train, X_test, y_test, n_features, 
                     experiment_name, data_dict, feature_names, scaler):
        """Train XGBoost with fixed hyperparameters"""
        start_time = time.time()
        
        try:
            # Fixed hyperparameters for fair comparison
            params = {
                'n_estimators': 300,
                'learning_rate': 0.05,
                'max_depth': 6,
                'min_child_weight': 1,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'reg_alpha': 0.1,
                'reg_lambda': 1.0,
                'random_state': self.random_state,
                'n_jobs': -1,
                'verbosity': 0
            }
            
            model = xgb.XGBRegressor(**params)
            model.fit(X_train, y_train)
            
            # Measure inference time
            inf_start = time.time()
            y_pred = model.predict(X_test)
            inference_time = time.time() - inf_start
            
            train_time = time.time() - start_time
            
            # Calculate model size
            model_size = 0.0
            try:
                with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as tmp:
                    tmp_path = tmp.name
                joblib.dump(model, tmp_path)
                model_size = os.path.getsize(tmp_path) / (1024 * 1024)
                os.unlink(tmp_path)
            except:
                model_size = 0.0
            
            # Count parameters (approximate for tree ensembles)
            n_trees = params['n_estimators']
            avg_leaves = 2 ** (params['max_depth'] - 1)
            n_params_approx = n_trees * avg_leaves
            
            # Save model ONLY if it's the final experiment (E5)
            if experiment_name == 'E5':
                self.save_final_model(model, 'XGBoost', data_dict, feature_names, scaler)
            
            return y_pred, train_time, inference_time, model_size, n_params_approx, model
            
        except Exception as e:
            print(f"     XGBoost training error: {e}")
            dummy_pred = np.zeros_like(y_test)
            return dummy_pred, time.time()-start_time, 0.0, 0.0, 0, None
    
    def train_tabnet(self, X_train, y_train, X_test, y_test, n_features,
                    experiment_name, data_dict, feature_names, scaler):
        """Train TabNet with fixed hyperparameters"""
        start_time = time.time()
        
        try:
            # Fixed hyperparameters for fair comparison
            model = TabNetRegressor(
                n_d=16,
                n_a=16,
                n_steps=3,
                gamma=1.3,
                lambda_sparse=1e-3,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=2e-2),
                mask_type='entmax',
                scheduler_params={
                    "mode": "min",
                    "patience": 10,
                    "min_lr": 1e-5,
                    "factor": 0.5,
                },
                scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
                verbose=0,
                seed=self.random_state
            )
            
            model.fit(
                X_train, y_train.reshape(-1, 1),
                eval_set=[(X_test, y_test.reshape(-1, 1))],
                max_epochs=100,
                patience=15,
                batch_size=256,
                virtual_batch_size=64,
                eval_metric=['rmse']
            )
            
            inf_start = time.time()
            y_pred = model.predict(X_test).flatten()
            inference_time = time.time() - inf_start
            
            train_time = time.time() - start_time
            
            # Model size
            model_size = 0.0
            try:
                with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp:
                    tmp_path = tmp.name
                model.save_model(tmp_path)
                model_size = os.path.getsize(tmp_path) / (1024 * 1024)
                os.unlink(tmp_path)
            except:
                model_size = 0.0
            
            # Approximate parameter count for TabNet
            n_params_approx = (n_features * 32 * 2 +
                             32 * 32 * 3 * 2 +
                             32 * 1)
            
            # Save model ONLY if it's the final experiment (E5)
            if experiment_name == 'E5':
                self.save_final_model(model, 'TabNet', data_dict, feature_names, scaler)
            
            return y_pred, train_time, inference_time, model_size, n_params_approx, model
            
        except Exception as e:
            print(f"     TabNet training error: {e}")
            dummy_pred = np.zeros_like(y_test)
            return dummy_pred, time.time()-start_time, 0.0, 0.0, 0, None

# ======================================================
# 5. STRICT VALIDATION FUNCTIONS
# ======================================================

def validate_feature_groups(feature_groups, train_df, test_df):
    """
    Perform mandatory validation checks on feature groups.
    """
    print("\n" + "="*60)
    print(" VALIDATING FEATURE GROUPS")
    print("="*60)
    
    all_features = []
    validation_passed = True
    
    # 1. Check feature exclusivity (no overlap between groups)
    print("\n Checking feature exclusivity...")
    for group_name, features in feature_groups.items():
        for other_group, other_features in feature_groups.items():
            if group_name != other_group:
                overlap = set(features) & set(other_features)
                if overlap:
                    print(f"    OVERLAP DETECTED between {group_name} and {other_group}:")
                    print(f"      Overlapping features: {list(overlap)[:5]}...")
                    validation_passed = False
        all_features.extend(features)
    
    # 2. Check existence in datasets
    print("\n Checking feature existence in datasets...")
    missing_in_train = []
    missing_in_test = []
    
    for feature in all_features:
        if feature not in train_df.columns:
            missing_in_train.append(feature)
        if feature not in test_df.columns:
            missing_in_test.append(feature)
    
    if missing_in_train:
        print(f"    {len(missing_in_train)} features missing in train set (will be filtered out)")
    else:
        print("    All features exist in train set")
    
    if missing_in_test:
        print(f"     {len(missing_in_test)} features missing in test set (will be filtered out)")
    else:
        print("    All features exist in test set")
    
    # 3. Check for duplicates in all_features
    print("\n Checking for duplicate features...")
    feature_counts = {}
    for feature in all_features:
        feature_counts[feature] = feature_counts.get(feature, 0) + 1
    
    duplicates = [feat for feat, count in feature_counts.items() if count > 1]
    if duplicates:
        print(f"    {len(duplicates)} duplicate features found:")
        print(f"      Sample duplicates: {duplicates[:5]}...")
        validation_passed = False
    else:
        print("    No duplicate features found")
    
    # 4. Print summary statistics
    print("\n Feature group statistics:")
    for group_name, features in feature_groups.items():
        existing_features = [f for f in features if f in train_df.columns and f in test_df.columns]
        print(f"   {group_name}: {len(existing_features)}/{len(features)} features (existing/total)")
    
    print(f"\n   Total unique features: {len(set(all_features))}")
    
    if validation_passed:
        print("\n ALL VALIDATION CHECKS PASSED!")
    else:
        print("\n  Validation warnings detected (will attempt to continue)")
    
    print("="*60)
    
    return validation_passed, set(all_features)

def validate_experiment_progression(experiments):
    """
    Validate that experiments follow monotonic progression: E0 ⊂ E1 ⊂ ... ⊂ E5
    """
    print("\n" + "="*60)
    print(" VALIDATING EXPERIMENT PROGRESSION")
    print("="*60)
    
    validation_passed = True
    prev_experiment = None
    prev_features = set()
    
    for exp_name in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']:
        if exp_name not in experiments:
            print(f"    Missing experiment: {exp_name}")
            validation_passed = False
            continue
        
        current_features = set(experiments[exp_name])
        
        # Check if experiment is subset of next experiment
        if prev_experiment:
            if not prev_features.issubset(current_features):
                missing = prev_features - current_features
                print(f"     {prev_experiment} is NOT a subset of {exp_name}")
                print(f"      Missing features: {list(missing)[:5]}...")
        
        # Check monotonic increase in feature count
        if prev_experiment:
            if len(current_features) <= len(prev_features):
                print(f"     Feature count not increasing: {exp_name} ({len(current_features)}) <= {prev_experiment} ({len(prev_features)})")
            else:
                print(f"    Feature count increased: {len(prev_features)} → {len(current_features)} (+{len(current_features) - len(prev_features)})")
        
        prev_experiment = exp_name
        prev_features = current_features
    
    print("\n EXPERIMENT PROGRESSION:")
    print(f"   E0 ({len(set(experiments['E0']))})")
    print(f"   E1 ({len(set(experiments['E1']))})")
    print(f"   E2 ({len(set(experiments['E2']))})")
    print(f"   E3 ({len(set(experiments['E3']))})")
    print(f"   E4 ({len(set(experiments['E4']))})")
    print(f"   E5 ({len(set(experiments['E5']))})")
    
    print("="*60)
    
    return True  # Always continue even with warnings

# ======================================================
# 6. ABLATION EXPERIMENT PIPELINE (CORRECTED)
# ======================================================

def run_ablation_experiments(train_df, test_df, 
                           models_to_use=['XGBoost', 'TabNet'], 
                           output_dir='ablation_results_T2'):
    """
    Main function to run all ablation experiments with strict validation.
    """
    print("="*80)
    print("STARTING COMPREHENSIVE ABLATION STUDY")
    print("="*80)
    print(" Using automatic feature detection from dataset")
    print(" Final models (E5 only) will be saved for SHAP analysis")
    print("="*80)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # ======================================================
    # STEP 1: Define feature groups automatically from data
    # ======================================================
    print("\n DETECTING FEATURE GROUPS FROM DATASET...")
    feature_groups = define_feature_groups_explicit(train_df)
    
    # ======================================================
    # STEP 2: Validate feature groups
    # ======================================================
    validation_passed, all_unique_features = validate_feature_groups(feature_groups, train_df, test_df)
    
    # ======================================================
    # STEP 3: Define experiments with cumulative progression
    # ======================================================
    print("\n DEFINING ABLATION EXPERIMENTS...")
    
    # Filter features to only those that exist in both datasets
    for group_name in feature_groups:
        feature_groups[group_name] = [f for f in feature_groups[group_name] 
                                     if f in train_df.columns and f in test_df.columns]
    
    # E0 = G0
    E0 = feature_groups['G0']
    
    # E1 = G0 + G1
    E1 = feature_groups['G0'] + feature_groups['G1']
    
    # E2 = G0 + G1 + G2
    E2 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2']
    
    # E3 = G0 + G1 + G2 + G3
    E3 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2'] + feature_groups['G3']
    
    # E4 = G0 + G1 + G2 + G3 + G4
    E4 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2'] + feature_groups['G3'] + feature_groups['G4']
    
    # E5 = G0 + G1 + G2 + G3 + G4 + G5
    E5 = feature_groups['G0'] + feature_groups['G1'] + feature_groups['G2'] + feature_groups['G3'] + feature_groups['G4'] + feature_groups['G5']
    
    # Remove duplicates while preserving order
    experiments = {
        'E0': list(dict.fromkeys(E0)),
        'E1': list(dict.fromkeys(E1)),
        'E2': list(dict.fromkeys(E2)),
        'E3': list(dict.fromkeys(E3)),
        'E4': list(dict.fromkeys(E4)),
        'E5': list(dict.fromkeys(E5))
    }
    
    # ======================================================
    # STEP 4: Validate experiment progression
    # ======================================================
    progression_passed = validate_experiment_progression(experiments)
    
    # ======================================================
    # STEP 5: Print experiment summary
    # ======================================================
    print("\n EXPERIMENT SUMMARY:")
    exp_display_names = {}
    for exp_name in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']:
        features = experiments[exp_name]
        
        # Create display name
        if exp_name == 'E0':
            display_name = 'E0 (G0 only)'
        elif exp_name == 'E1':
            display_name = 'E1 (G0+G1)'
        elif exp_name == 'E2':
            display_name = 'E2 (G0+G1+G2)'
        elif exp_name == 'E3':
            display_name = 'E3 (G0+G1+G2+G3)'
        elif exp_name == 'E4':
            display_name = 'E4 (G0+G1+G2+G3+G4)'
        elif exp_name == 'E5':
            display_name = 'E5 (All features)'
        
        exp_display_names[exp_name] = display_name
        
        print(f"  {display_name}")
        print(f"    Features: {len(features)}")
        if features:
            print(f"    Sample features: {features[:3]}...")
        else:
            print(f"      No features available!")
    
    # ======================================================
    # STEP 6: Run experiments
    # ======================================================
    print(f"\n MODELS TO TRAIN: {models_to_use}")
    print(" Only E5 models will be saved for SHAP analysis")
    
    # Initialize results storage
    all_results = []
    trainer = AblationModelTrainer()
    
    # Run each experiment
    for exp_name in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']:
        print(f"\n{'='*60}")
        print(f"🏃‍♂️ RUNNING EXPERIMENT: {exp_display_names[exp_name]}")
        print(f"{'='*60}")
        
        # Get features for this experiment
        exp_features = experiments[exp_name]
        
        if len(exp_features) == 0:
            print(f"    No features available for this experiment, skipping...")
            continue
        
        # Prepare data for this experiment
        data = prepare_ablation_data(train_df, test_df, exp_features)
        
        if data is None:
            print(f"   Failed to prepare data for experiment {exp_name}")
            continue
        
        print(f"  Features: {len(exp_features)}")
        print(f"  Train samples: {len(data['y_train'])}")
        print(f"  Test samples: {len(data['y_test'])}")
        
        # Train each model
        for model_name in models_to_use:
            print(f"\n  Training {model_name}...")
            
            try:
                if model_name == 'XGBoost':
                    y_pred, train_time, inf_time, model_size, n_params, model = \
                        trainer.train_xgboost(data['X_train'], data['y_train'],
                                            data['X_test'], data['y_test'],
                                            data['n_features'],
                                            exp_name, data,
                                            data['feature_names'], data['scaler'])
                
                elif model_name == 'TabNet':
                    y_pred, train_time, inf_time, model_size, n_params, model = \
                        trainer.train_tabnet(data['X_train'], data['y_train'],
                                           data['X_test'], data['y_test'],
                                           data['n_features'],
                                           exp_name, data,
                                           data['feature_names'], data['scaler'])
                
                else:
                    print(f"     Unknown model: {model_name}")
                    continue
                
                # Check if we got valid predictions
                if y_pred is None or len(y_pred) == 0:
                    print(f"     No predictions returned for {model_name}")
                    continue
                
                # Calculate metrics
                metrics = calculate_comprehensive_metrics(
                    data['y_test'], y_pred, model_name, train_time, inf_time, 
                    model_size, len(exp_features), n_params
                )
                
                # Add experiment info
                metrics['Experiment'] = exp_display_names[exp_name]
                metrics['Experiment_Code'] = exp_name
                
                all_results.append(metrics)
                
                print(f"     RMSE: {metrics['RMSE']:.4f}, R²: {metrics['R²']:.4f}")
                print(f"      Train: {train_time:.1f}s, Infer: {inf_time:.3f}s")
                print(f"     Features: {len(exp_features)}")
                
                # Special message for E5 experiment
                if exp_name == 'E5':
                    print(f"     Model saved for SHAP analysis!")
                
            except Exception as e:
                print(f"     Failed: {str(e)}")
                import traceback
                traceback.print_exc()
    
    # Check if we have results
    if not all_results:
        print("\n WARNING: No results collected!")
        return pd.DataFrame(), experiments, exp_display_names
    
    # ======================================================
    # STEP 7: Create and save results
    # ======================================================
    results_df = pd.DataFrame(all_results)
    
    # Reorder columns for better readability
    available_columns = results_df.columns.tolist()
    column_order = ['Experiment', 'Experiment_Code', 'Model', 'Num Features', 
                   'RMSE', 'MAE', 'R²', 'MAPE (%)', 'Index of Agreement (d)', 
                   'RPD', 'Training Time (s)', 'Inference Time (s)', 
                   'Num Parameters', 'MSE', 'Explained Variance',
                   'MBE', 'Std of Residuals', 'Coverage 95%',  'RPIQ']
    
    # Keep only columns that exist
    column_order = [col for col in column_order if col in available_columns]
    # Add any remaining columns
    remaining_cols = [col for col in available_columns if col not in column_order]
    column_order.extend(remaining_cols)
    
    results_df = results_df[column_order]
    
    # Save results
    results_path = os.path.join(output_dir, 'ablation_results.csv')
    results_df.to_csv(results_path, index=False, float_format='%.6f')
    
    # Save experiment configuration
    config_path = os.path.join(output_dir, 'experiment_configuration.json')
    with open(config_path, 'w') as f:
        config_data = {
            'feature_groups': {k: len(v) for k, v in feature_groups.items()},
            'experiments': {k: len(v) for k, v in experiments.items()},
            'total_unique_features': len(all_unique_features),
            'timestamp': time.strftime("%Y-%m-%d %H:%M:%S")
        }
        json.dump(config_data, f, indent=4)
    
    print(f"\n{'='*80}")
    print(f" ABLATION STUDY COMPLETED")
    print(f" Results saved to: {results_path}")
    print(f" Configuration saved to: {config_path}")
    print(f" FINAL MODELS SAVED IN: final_models_for_shap_T2/")
    print(f" Total results: {len(results_df)} rows")
    print(f"{'='*80}")
    
    # Print final summary table
    print("\n FINAL RESULTS SUMMARY:")
    summary_df = results_df[['Experiment', 'Model', 'Num Features', 'RMSE', 'R²']].copy()
    print(summary_df.to_string(index=False))
    
    return results_df, experiments, exp_display_names

# ======================================================
# 7. COMPLETE MODEL LOADING FUNCTION (FULLY RESTORED)
# ======================================================

def load_final_models_for_shap():
    """
    Load saved final models for SHAP analysis.
    Returns a dictionary with models, scalers, feature names, and test data.
    """
    print("\n" + "="*80)
    print(" LOADING FINAL MODELS FOR SHAP ANALYSIS")
    print("="*80)
    
    base_dir = 'final_models_for_shap_T2'
    
    if not os.path.exists(base_dir):
        print(f" Directory not found: {base_dir}")
        print("   Please run the ablation study first to train and save models.")
        return None
    
    models_dict = {}
    
    # List all model files
    model_files = [f for f in os.listdir(base_dir) if f.endswith(('_model.pkl', '_model.zip'))]
    
    if not model_files:
        print(" No model files found in final_models_for_shap/")
        return None
    
    print(f"\n Found {len(model_files)} model files:")
    
    for model_file in model_files:
        if model_file.endswith('_model.pkl'):
            model_name = model_file.replace('_model.pkl', '')
        elif model_file.endswith('_model.zip'):
            model_name = model_file.replace('_model.zip', '')
        else:
            continue
        
        print(f"\n Loading {model_name}...")
        
        try:
            # Load the model
            if model_name == 'XGBoost':
                model_path = os.path.join(base_dir, f'{model_name}_model.pkl')
                model = joblib.load(model_path)
                print(f"    Model loaded from: {model_path}")
                
            elif model_name == 'TabNet':
                model_path = os.path.join(base_dir, f'{model_name}_model.zip')
                model = TabNetRegressor()
                model.load_model(model_path)
                print(f"    Model loaded from: {model_path}")
            
            # Load scaler
            scaler_path = os.path.join(base_dir, f'{model_name}_scaler.pkl')
            scaler = joblib.load(scaler_path)
            print(f"    Scaler loaded from: {scaler_path}")
            
            # Load feature names
            features_path = os.path.join(base_dir, f'{model_name}_features.json')
            with open(features_path, 'r') as f:
                features_info = json.load(f)
            
            feature_names = features_info['feature_names']
            print(f"    Feature names loaded: {len(feature_names)} features")
            
            # Load test data
            test_data_path = os.path.join(base_dir, f'{model_name}_test_data.pkl')
            test_data = joblib.load(test_data_path)
            print(f"    Test data loaded: {len(test_data['X_test'])} samples")
            
            # Store everything in dictionary
            models_dict[model_name] = {
                'model': model,
                'scaler': scaler,
                'feature_names': feature_names,
                'test_data': test_data,
                'X_test_raw': test_data['X_test'],
                'X_test_scaled': test_data['X_test_scaled'],
                'y_test': test_data['y_test']
            }
            
            print(f"    {model_name} ready for SHAP analysis!")
            
        except Exception as e:
            print(f"    Error loading {model_name}: {e}")
    
    print(f"\n" + "="*80)
    print(f" LOADED {len(models_dict)} MODELS FOR SHAP ANALYSIS")
    print("="*80)
    
    # Print summary
    for model_name, model_info in models_dict.items():
        print(f"\n {model_name}:")
        print(f"   Features: {len(model_info['feature_names'])}")
        print(f"   Test samples: {len(model_info['y_test'])}")
        print(f"   Sample features: {model_info['feature_names'][:3]}...")
    
    return models_dict

# ======================================================
# 8. RUN SHAP ANALYSIS ON FINAL MODELS (FULLY RESTORED)
# ======================================================

def run_shap_analysis_on_final_models():
    """
    Run SHAP analysis on the final saved models.
    """
    print("\n" + "="*80)
    print(" RUNNING SHAP ANALYSIS ON FINAL MODELS")
    print("="*80)
    
    # Load models
    models_dict = load_final_models_for_shap()
    
    if not models_dict:
        print(" No models loaded. Cannot run SHAP analysis.")
        return
    
    # Check if SHAP is installed
    try:
        import shap
        print(" SHAP library is available")
    except ImportError:
        print(" SHAP not installed. Please install it first:")
        print("   pip install shap")
        return
    
    # Create directory for SHAP outputs
    shap_output_dir = 'final_models_for_shap_T2/shap_results'
    os.makedirs(shap_output_dir, exist_ok=True)
    
    # Run SHAP for each model
    for model_name, model_info in models_dict.items():
        print(f"\n{'='*60}")
        print(f" Running SHAP analysis for {model_name}")
        print(f"{'='*60}")
        
        model = model_info['model']
        feature_names = model_info['feature_names']
        X_test_scaled = model_info['X_test_scaled']
        
        print(f"  Features: {len(feature_names)}")
        print(f"  Test samples: {X_test_scaled.shape[0]}")
        
        # Sample data for faster SHAP computation
        sample_size = min(100, X_test_scaled.shape[0])
        X_sample = X_test_scaled[:sample_size]
        
        print(f"  Using {sample_size} samples for SHAP computation")
        
        try:
            if model_name == 'XGBoost':
                # Tree-based models use TreeExplainer
                print(f"  Using TreeExplainer for {model_name}")
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_sample)
                
                # Summary plot
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_values, X_sample, 
                                 feature_names=feature_names, 
                                 show=False, plot_size=None)
                plt.title(f'SHAP Summary Plot - {model_name}', fontsize=16)
                plt.tight_layout()
                summary_plot_path = os.path.join(shap_output_dir, f'shap_summary_{model_name}.png')
                plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"   Summary plot saved: {summary_plot_path}")
                
                # Bar plot (mean absolute SHAP values)
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_values, X_sample, 
                                 feature_names=feature_names, 
                                 plot_type="bar", show=False)
                plt.title(f'SHAP Feature Importance - {model_name}', fontsize=16)
                plt.tight_layout()
                bar_plot_path = os.path.join(shap_output_dir, f'shap_bar_{model_name}.png')
                plt.savefig(bar_plot_path, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"   Bar plot saved: {bar_plot_path}")
                
                # Calculate and save feature importance
                shap_values_abs = np.abs(shap_values).mean(0)
                feature_importance = pd.DataFrame({
                    'feature': feature_names,
                    'importance': shap_values_abs
                }).sort_values('importance', ascending=False)
                
                importance_path = os.path.join(shap_output_dir, f'feature_importance_{model_name}.csv')
                feature_importance.to_csv(importance_path, index=False)
                print(f"   Feature importance saved: {importance_path}")
                
                # Top 10 features
                print(f"\n   Top 10 important features for {model_name}:")
                for i, row in feature_importance.head(10).iterrows():
                    print(f"     {i+1:2d}. {row['feature']}: {row['importance']:.6f}")
                
            elif model_name == 'TabNet':
                # Neural network models use KernelExplainer
                print(f"  Using KernelExplainer for {model_name}")
                
                # Define prediction function
                def predict_fn(X):
                    return model.predict(X).flatten()
                
                # Use a small background dataset
                background = shap.sample(X_sample, 10)
                explainer = shap.KernelExplainer(predict_fn, background)
                
                # Calculate SHAP values for a small sample
                shap_values = explainer.shap_values(X_sample[:50], nsamples=100)
                
                # Summary plot
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_values, X_sample[:50], 
                                 feature_names=feature_names, 
                                 show=False, plot_size=None)
                plt.title(f'SHAP Summary Plot - {model_name}', fontsize=16)
                plt.tight_layout()
                summary_plot_path = os.path.join(shap_output_dir, f'shap_summary_{model_name}.png')
                plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"   Summary plot saved: {summary_plot_path}")
                
                # Calculate and save feature importance
                shap_values_abs = np.abs(shap_values).mean(0)
                feature_importance = pd.DataFrame({
                    'feature': feature_names,
                    'importance': shap_values_abs
                }).sort_values('importance', ascending=False)
                
                importance_path = os.path.join(shap_output_dir, f'feature_importance_{model_name}.csv')
                feature_importance.to_csv(importance_path, index=False)
                print(f"  Feature importance saved: {importance_path}")
                
                # Top 10 features
                print(f"\n   Top 10 important features for {model_name}:")
                for i, row in feature_importance.head(10).iterrows():
                    print(f"     {i+1:2d}. {row['feature']}: {row['importance']:.6f}")
            
            print(f"\n   SHAP analysis complete for {model_name}")
            
        except Exception as e:
            print(f"   Error in SHAP analysis for {model_name}: {e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n" + "="*80)
    print(" SHAP ANALYSIS COMPLETED!")
    print(f" All results saved in: {shap_output_dir}/")
    print("="*80)
    
    # Create a README file for the SHAP results
    readme_content = f"""# SHAP Analysis Results

This directory contains SHAP analysis results for the final models.

## Generated Files:
For each model, you will find:
1. `shap_summary_[model_name].png` - SHAP summary plot
2. `shap_bar_[model_name].png` - SHAP bar plot (feature importance)
3. `feature_importance_[model_name].csv` - Feature importance values

## Models Analyzed:
{', '.join(models_dict.keys())}

## How to Interpret:
1. **Summary Plot**: Shows the impact of each feature on model output
2. **Bar Plot**: Shows mean absolute SHAP values (feature importance)
3. **CSV File**: Contains numerical importance values for each feature

## Date Generated: {time.strftime("%Y-%m-%d %H:%M:%S")}
"""
    
    readme_path = os.path.join(shap_output_dir, 'README.md')
    with open(readme_path, 'w') as f:
        f.write(readme_content)
    
    print(f"\n Documentation saved: {readme_path}")

# ======================================================
# 9. MAIN EXECUTION SCRIPT (NO PROMPTS, BOTH MODELS)
# ======================================================

def main():
    """
    Main execution function for the ablation study with model saving.
    NO INTERACTIVE PROMPTS - RUNS AUTOMATICALLY
    """
    print("="*80)
    print(" CORN YIELD ESTIMATION - ABLATION STUDY")
    print("="*80)
    print("Running automatically with NO interactive prompts...")
    print("Training BOTH XGBoost and TabNet models...")
    print("="*80)
    
    # Set your paths here - USE YOUR ACTUAL PATHS
    train_path = "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_train_FE.csv"
    test_path = "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_test_FE.csv"
    
    # Load your feature-engineered data
    print("\n LOADING DATA...")
    try:
        train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)
        
        print(f" Train shape: {train_df.shape}")
        print(f" Test shape: {test_df.shape}")
        
        # Check for required columns
        if 'yield' not in train_df.columns or 'yield' not in test_df.columns:
            print(" ERROR: 'yield' column not found in datasets!")
            print("   Available columns in train:")
            print(f"   {train_df.columns.tolist()[:10]}...")
            return
        
        print(" Data loaded successfully!")
        
    except Exception as e:
        print(f" Error loading data: {e}")
        print("Please check your file paths and ensure the CSV files exist.")
        return
    
    # Define which models to use - BOTH MODELS
    models_to_use = ['XGBoost', 'TabNet']
    
    print(f"\n MODELS TO TRAIN: {models_to_use}")
    print(" Only E5 models will be saved for SHAP analysis")
    
    # Run ablation experiments
    try:
        start_time = time.time()
        results_df, experiments, exp_display_names = run_ablation_experiments(
            train_df, test_df, 
            models_to_use=models_to_use,
            output_dir='ablation_results_T2'
        )
        
        execution_time = time.time() - start_time
        
        # Final summary
        print("\n" + "="*80)
        print(" ABLATION STUDY COMPLETE")
        print("="*80)
        print(f"  Total execution time: {execution_time:.1f} seconds")
        
        # Check if files were saved
        print("\n CHECKING SAVED FILES:")
        
        # Check ablation results
        if os.path.exists('ablation_results_T2/ablation_results.csv'):
            results_size = os.path.getsize('ablation_results_T2/ablation_results.csv') / 1024
            print(f"   ablation_results.csv ({results_size:.1f} KB)")
            
            # Load and display results
            saved_results = pd.read_csv('ablation_results_T2/ablation_results.csv')
            print(f"\n SAVED RESULTS SUMMARY:")
            print(f"  Total experiments: {len(saved_results)}")
            print(f"  Unique models: {saved_results['Model'].unique().tolist()}")
            print(f"  Experiments: {saved_results['Experiment'].unique().tolist()}")
        else:
            print("   ablation_results.csv NOT FOUND!")
        
        # Check saved models
        model_files = []
        if os.path.exists('final_models_for_shap_T2'):
            model_files = os.listdir('final_models_for_shap_T2')
        
        if model_files:
            print(f"\n   final_models_for_shap_T2/ directory contains {len(model_files)} files")
            xgb_files = [f for f in model_files if 'XGBoost' in f]
            tabnet_files = [f for f in model_files if 'TabNet' in f]
            print(f"     XGBoost files: {len(xgb_files)}")
            print(f"     TabNet files: {len(tabnet_files)}")
            
            for file in sorted(model_files):
                if any(x in file for x in ['_model.', '_scaler.', '_features.', '_test_data.']):
                    file_size = os.path.getsize(f'final_models_for_shap_T2/{file}') / 1024
                    print(f"     - {file} ({file_size:.1f} KB)")
        else:
            print("   No files found in final_models_for_shap/")
        
        # Summary message
        print("\n" + "="*80)
        print(" SUCCESS: Ablation study completed successfully!")
        print(f" Results saved in: ablation_results_T2/ablation_results.csv")
        print(f" Models saved in: final_models_for_shap_T2/")
        print("\n To run SHAP analysis later, use:")
        print("   run_shap_analysis_on_final_models()")
        print("="*80)
        
    except Exception as e:
        print(f"\n ERROR in ablation study: {e}")
        import traceback
        traceback.print_exc()

# ======================================================
# 10. DIRECT EXECUTION
# ======================================================

if __name__ == "__main__":
    # Create necessary directories
    os.makedirs('ablation_results_T2', exist_ok=True)
    os.makedirs('final_models_for_shap_T2', exist_ok=True)
    
    # Run the ablation study
    print("Starting automatic ablation study...")
    print("Training BOTH XGBoost and TabNet models...")
    print("NO interactive prompts - running fully automatically")
    print("-" * 50)
    
    main()
    print("\n Ablation study execution completed!")

SHAP for diagnostics

SHAP

In [ ]:
"""
COMPLETE SHAP ANALYSIS FOR XGBOOST MODELS (TRACK 1 & TRACK 2) - FIXED VERSION

Purpose: Post-hoc interpretability of saved XGBoost models using SHAP
"""

import os
import sys
import numpy as np
import pandas as pd
import joblib
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# SHAP
import shap

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# ======================================================
# 1. CONFIGURATION - UPDATE THESE PATHS FOR YOUR SYSTEM
# ======================================================

CONFIG = {
    # Model paths (from your saved models)
    'track1_model_path': 'E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_for_shap/XGBoost_model.pkl',  # Track 1 model
    'track2_model_path': 'E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_for_shap_T2/XGBoost_model.pkl',  # Track 2 model
    
    # Feature info paths (from your saved models)
    'track1_features_path': 'E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_for_shap/XGBoost_features.json',
    'track2_features_path': 'E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_for_shap_T2/XGBoost_features.json',
    
    # Data paths (from your dataset)
    'train_path': "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_train_FE.csv",
    'test_path': "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_test_FE.csv",
    
    # Output directories
    'output_dir': 'shap_analysis_results_complete',
    'track1_output_dir': 'shap_analysis_results_complete/track1',
    'track2_output_dir': 'shap_analysis_results_complete/track2',
    
    # SHAP parameters
    'n_samples_for_shap': 1000,  # Number of samples for SHAP computation
    'top_n_features': 20,  # Top N features to show in plots
    
    # Plotting style
    'style': 'seaborn-v0_8-whitegrid',
    'color_palette': 'viridis',
    'dpi': 300,
}

# ======================================================
# 2. FEATURE GROUP DEFINITION
# ======================================================

def define_feature_groups(df):
    """
    Define feature groups based on your engineering pipeline.
    Returns a dictionary mapping group names to feature lists.
    This matches your feature engineering groups.
    """
    groups = {}
    
    # G0: Raw monthly features (baseline)
    g0_features = []
    for var in ["NDVI", "GPP", "PPT", "TMEAN", "TMIN", "TMAX", "TDMEAN", "VPDMAX"]:
        monthly_cols = [col for col in df.columns 
                       if col.startswith(f"{var}_") and col.split('_')[-1].isdigit()]
        g0_features.extend(monthly_cols)
    groups['G0'] = sorted(list(set(g0_features)))
    
    # G1: Seasonal aggregates
    g1_features = []
    for col in df.columns:
        if 'season' in col.lower():
            if any(stat in col.lower() for stat in ['_season_mean', '_season_sum', '_season_std', '_season_max']):
                g1_features.append(col)
    groups['G1'] = g1_features     
            
    # G2: Phenological phase features
    g2_features = []
    for col in df.columns:
        if any(phase in col.lower() for phase in ['early', 'peak', 'late']):
            g2_features.append(col)
    groups['G2'] = g2_features
    
    # G3: Climate stress indicators
    g3_features = []
    for col in df.columns:
        if any(term in col.lower() for term in ['range', 'stress', 'heat']):
            g3_features.append(col)
    groups['G3'] = g3_features
    
    # G4: Efficiency features
    g4_features = []
    for col in df.columns:
        if any(term in col.lower() for term in ['efficiency', 'ndvi_ppt', 'gpp_ppt']):
            g4_features.append(col)
    groups['G4'] = g4_features
    
    # G5: Spatial anomaly features
    g5_features = [col for col in df.columns if col.endswith('_z')]
    groups['G5'] = g5_features
    
    # Gs: STATIC FEATURES (for Track 2 only)
    gs_features = []
    
    # Soil properties (from your sample data)
    soil_properties = ['awc', 'aws', 'b_density', 'cec', 'clay_percent', 
                      'field_capacity', 'organic_matter', 'pH', 'saturated_hc', 
                      'sand_percent', 'wilting_point']  
    
    # Geographic features
    geographic_features = ['X', 'Y']  # Longitude, Latitude
    
    # Check which static features exist in the dataframe
    for feature in soil_properties + geographic_features:
        if feature in df.columns:
            gs_features.append(feature)
    
    # Also check for any features that don't match dynamic patterns
    dynamic_patterns = ['_1', '_2', '_3', '_4', '_5', '_6', '_7', '_8', '_9', '_10', '_11', '_12',
                       'season', 'early', 'peak', 'late', 'efficiency', 'z', 'range', 'stress']
    
    for col in df.columns:
        if col not in gs_features and col not in g0_features + g1_features + g2_features + g3_features + g4_features + g5_features:
            # Check if it looks like a static feature (no numbers, not in dynamic patterns)
            is_dynamic = any(pattern in col for pattern in dynamic_patterns)
            if not is_dynamic and col not in ['yield', 'year', 'STATE', 'GEOID']:
                gs_features.append(col)
    
    groups['Gs'] = sorted(list(set(gs_features)))
    
    # Remove duplicates and ensure features exist
    for group in groups:
        groups[group] = [f for f in groups[group] if f in df.columns]
        groups[group] = sorted(set(groups[group]))
    
    # Print summary
    print(f"\n Feature group summary:")
    for group, features in groups.items():
        print(f"  {group}: {len(features)} features")
        if features:
            print(f"    Sample: {features[:3]}")
    
    return groups

# ======================================================
# 3. LOAD SAVED MODEL INFORMATION
# ======================================================

def load_model_and_features(model_path, features_path, track_name):
    """
    Load saved XGBoost model and its feature information.
    
    Parameters:
    -----------
    model_path : str
        Path to saved model file
    features_path : str
        Path to saved features JSON file
    track_name : str
        Name of track for logging
    
    Returns:
    --------
    dict : Contains model, feature_names, and metadata
    """
    print(f"\n Loading XGBoost model and features for {track_name}...")
    
    results = {}
    
    try:
        # 1. Load the model
        print(f"  Loading model from: {model_path}")
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            results['model'] = model
            print(f"  Model loaded successfully")
            print(f"  Model type: {model.__class__.__name__}")
            
            # Check model feature count
            if hasattr(model, 'feature_importances_'):
                print(f"  Model expects {len(model.feature_importances_)} features")
        else:
            print(f"  Model file not found: {model_path}")
            return None
    
    except Exception as e:
        print(f"  Error loading model: {e}")
        return None
    
    try:
        # 2. Load feature information
        print(f"  Loading feature info from: {features_path}")
        if os.path.exists(features_path):
            with open(features_path, 'r') as f:
                features_info = json.load(f)
            
            results['feature_info'] = features_info
            results['feature_names'] = features_info.get('feature_names', [])
            results['n_features'] = features_info.get('n_features', 0)
            results['experiment_name'] = features_info.get('experiment_name', 'unknown')
            
            print(f"  Feature info loaded successfully")
            print(f"  Features in saved model: {len(results['feature_names'])}")
            print(f"  Experiment: {results['experiment_name']}")
            print(f"  Sample features: {results['feature_names'][:5]}")
        else:
            print(f"  Feature info file not found: {features_path}")
            # Try to get features from model
            if hasattr(model, 'get_booster'):
                try:
                    booster = model.get_booster()
                    feature_names = booster.feature_names
                    if feature_names:
                        results['feature_names'] = feature_names
                        results['n_features'] = len(feature_names)
                        print(f"  Extracted {len(feature_names)} features from model")
                except:
                    pass
    
    except Exception as e:
        print(f"   Error loading feature info: {e}")
    
    return results

# ======================================================
# 4. DATA PREPARATION WITH FEATURE ALIGNMENT
# ======================================================

def prepare_data_with_feature_alignment(train_path, test_path, model_feature_names, track):
    """
    Load data and align features with model's expected features.
    
    Parameters:
    -----------
    train_path : str
        Path to training data CSV
    test_path : str
        Path to test data CSV
    model_feature_names : list
        Features expected by the model
    track : str
        'track1' or 'track2'
    
    Returns:
    --------
    dict : Contains aligned X_train, X_test, y_train, y_test, feature_groups
    """
    print(f"\n Loading and aligning data for {track}...")
    
    # Load data
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    print(f"  Original train shape: {train_df.shape}")
    print(f"  Original test shape: {test_df.shape}")
    
    # Define feature groups from the data
    feature_groups = define_feature_groups(train_df)
    
    # Get all possible features based on track
    if track == 'track1':
        # Track 1: Dynamic features only (G0-G5)
        all_possible_features = []
        for group in ['G0', 'G1', 'G2', 'G3', 'G4', 'G5']:
            all_possible_features.extend(feature_groups.get(group, []))
    elif track == 'track2':
        # Track 2: Dynamic + Static features (G0-G5 + Gs)
        all_possible_features = []
        for group in ['G0', 'G1', 'G2', 'G3', 'G4', 'G5', 'Gs']:
            all_possible_features.extend(feature_groups.get(group, []))
    else:
        raise ValueError("track must be 'track1' or 'track2'")
    
    all_possible_features = sorted(list(set(all_possible_features)))
    print(f"  Possible features for {track}: {len(all_possible_features)}")
    
    # Align features with model's expected features
    if model_feature_names:
        print(f"  Model expects {len(model_feature_names)} features")
        
        # Find common features between model and data
        common_features = [f for f in model_feature_names if f in train_df.columns]
        missing_in_data = [f for f in model_feature_names if f not in train_df.columns]
        extra_in_data = [f for f in all_possible_features if f not in model_feature_names]
        
        print(f"  Common features: {len(common_features)}")
        print(f"  Features in model but missing in data: {len(missing_in_data)}")
        if missing_in_data:
            print(f"    Missing: {missing_in_data[:5]}")
        print(f"  Features in data but not in model: {len(extra_in_data)}")
        
        # Use model's feature order
        final_features = [f for f in model_feature_names if f in common_features]
        
        # Check if we have enough features
        if len(final_features) < len(model_feature_names):
            print(f"   Warning: Only {len(final_features)}/{len(model_feature_names)} features available")
            # Add missing features with zeros
            for f in missing_in_data:
                print(f"    Will create dummy feature: {f}")
                train_df[f] = 0.0
                test_df[f] = 0.0
                final_features.append(f)
    else:
        # Use all possible features
        final_features = all_possible_features
        print(f"  Using all {len(final_features)} possible features")
    
    # Ensure we have the features in the correct order
    # final_features = sorted(list(set(final_features)))
    final_features = [f for f in model_feature_names if f in train_df.columns]

    print(f"  Final feature count: {len(final_features)}")
    print(f"  Sample features: {final_features[:5]}")
    
    # Prepare feature matrices
    X_train = train_df[final_features].copy()
    X_test = test_df[final_features].copy()
    
    # Handle missing values
    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_test.mean())
    
    # Target variable
    y_train = train_df['yield'].values
    y_test = test_df['yield'].values
    
    print(f"  X_train aligned shape: {X_train.shape}")
    print(f"  X_test aligned shape: {X_test.shape}")
    
    return {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_names': final_features,
        'feature_groups': feature_groups,
        'track': track,
        'model_feature_names': model_feature_names
    }

# ======================================================
# 5. SHAP COMPUTATION WITH SAFETY CHECKS
# ======================================================

def compute_shap_values_safely(model, X_data, feature_names, n_samples=None):
    """
    Compute SHAP values with safety checks for feature alignment.
    
    Parameters:
    -----------
    model : XGBoost model
        Trained XGBoost model
    X_data : pandas DataFrame
        Data to compute SHAP values for
    feature_names : list
        Expected feature names
    n_samples : int or None
        Number of samples to use (None = all)
    
    Returns:
    --------
    explainer : SHAP TreeExplainer
    shap_values : numpy array
        SHAP values
    X_sample : pandas DataFrame
        Sampled data used for SHAP
    """
    print(" Computing SHAP values with safety checks...")
    
    # Convert to numpy array for SHAP
    if isinstance(X_data, pd.DataFrame):
        X_array = X_data.values
    else:
        X_array = X_data
    
    print(f"  Input data shape: {X_array.shape}")
    print(f"  Model expects features: {len(feature_names)}")
    
    # Check feature count match
    if X_array.shape[1] != len(feature_names):
        print(f"   Warning: Data has {X_array.shape[1]} features, model expects {len(feature_names)}")
        print(f"  Attempting to align features...")
        
        # If we have more features than model expects, use first n features
        if X_array.shape[1] > len(feature_names):
            print(f"  Using first {len(feature_names)} features")
            X_array = X_array[:, :len(feature_names)]
        # If we have fewer features, pad with zeros
        elif X_array.shape[1] < len(feature_names):
            print(f"  Padding with zeros to match {len(feature_names)} features")
            padding = np.zeros((X_array.shape[0], len(feature_names) - X_array.shape[1]))
            X_array = np.hstack([X_array, padding])
    
    print(f"  Aligned data shape: {X_array.shape}")
    
    # Sample data if specified
    if n_samples is not None and n_samples < len(X_array):
        print(f"  Sampling {n_samples} samples (out of {len(X_array)})")
        indices = np.random.choice(len(X_array), min(n_samples, len(X_array)), replace=False)
        X_sample_array = X_array[indices]
        if isinstance(X_data, pd.DataFrame):
            X_sample_df = X_data.iloc[indices]
        else:
            X_sample_df = None
    else:
        X_sample_array = X_array
        X_sample_df = X_data if isinstance(X_data, pd.DataFrame) else None
        print(f"  Using all {len(X_sample_array)} samples")
    
    print(f"  Final data shape for SHAP: {X_sample_array.shape}")
    
    try:
        # Create TreeExplainer
        print("  Creating TreeExplainer...")
        explainer = shap.TreeExplainer(model)
        
        # Compute SHAP values
        print("  Computing SHAP values...")
        shap_values = explainer.shap_values(X_sample_array)
        
        print(f"   SHAP computation successful!")
        print(f"  SHAP values shape: {shap_values.shape}")
        print(f"  Expected value (base value): {explainer.expected_value:.4f}")
        
        return explainer, shap_values, X_sample_df if X_sample_df is not None else X_sample_array
        
    except Exception as e:
        print(f"   Error computing SHAP values: {e}")
        
        # Try alternative approach
        print("  Trying alternative SHAP computation...")
        try:
            explainer = shap.TreeExplainer(model, X_sample_array[:100])  # Small background
            shap_values = explainer.shap_values(X_sample_array)
            
            print(f"  Alternative SHAP computation successful!")
            print(f"  SHAP values shape: {shap_values.shape}")
            
            return explainer, shap_values, X_sample_df if X_sample_df is not None else X_sample_array
            
        except Exception as e2:
            print(f"  Alternative also failed: {e2}")
            raise

# ======================================================
# 6. GROUP-LEVEL SHAP AGGREGATION
# ======================================================

def aggregate_shap_by_groups(shap_values, feature_names, feature_groups, track):
    """
    Aggregate feature-level SHAP values to group-level importance.
    
    Parameters:
    -----------
    shap_values : numpy array
        SHAP values (n_samples, n_features)
    feature_names : list
        List of feature names
    feature_groups : dict
        Dictionary mapping group names to feature lists
    track : str
        'track1' or 'track2'
    
    Returns:
    --------
    group_importance_df : pandas DataFrame
        DataFrame with group-level SHAP importance
    feature_importance_df : pandas DataFrame
        DataFrame with feature-level SHAP importance
    """
    print(" Aggregating SHAP values by feature groups...")
    
    # Calculate mean absolute SHAP per feature
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Create feature-level importance DataFrame
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'mean_abs_shap': mean_abs_shap,
        'group': 'Other'  # Default group
    })
    
    # Assign features to groups
    for group_name, group_features in feature_groups.items():
        # Only include groups relevant to this track
        if track == 'track1' and group_name == 'Gs':
            continue
            
        for feature in group_features:
            if feature in feature_names:
                feature_importance_df.loc[feature_importance_df['feature'] == feature, 'group'] = group_name
    
    # Calculate group-level importance
    group_importance = {}
    group_counts = {}
    group_features_dict = {}
    
    for group_name in feature_groups.keys():
        # Only include groups relevant to this track
        if track == 'track1' and group_name == 'Gs':
            continue
            
        # Get features in this group
        group_features = [f for f in feature_groups[group_name] if f in feature_names]
        group_features_dict[group_name] = group_features
        
        if group_features:
            # Get indices of these features
            indices = [feature_names.index(f) for f in group_features]
            # Calculate mean absolute SHAP for this group
            group_shap_values = mean_abs_shap[indices]
            group_importance[group_name] = group_shap_values.sum()
            group_counts[group_name] = len(group_features)
        else:
            group_importance[group_name] = 0.0
            group_counts[group_name] = 0
    
    # Create group importance DataFrame
    group_importance_df = pd.DataFrame({
        'group': list(group_importance.keys()),
        'total_shap_importance': list(group_importance.values()),
        'n_features': [group_counts[g] for g in group_importance.keys()],
        'features': [group_features_dict[g] for g in group_importance.keys()]
    })
    
    # Calculate normalized importance
    total_importance = group_importance_df['total_shap_importance'].sum()
    if total_importance > 0:
        group_importance_df['normalized_importance'] = group_importance_df['total_shap_importance'] / total_importance * 100
    else:
        group_importance_df['normalized_importance'] = 0.0
    
    # Sort by importance
    group_importance_df = group_importance_df.sort_values('total_shap_importance', ascending=False)
    feature_importance_df = feature_importance_df.sort_values('mean_abs_shap', ascending=False)
    
    print(f"  Found {len(group_importance_df)} feature groups")
    print(f"  Total SHAP importance: {total_importance:.4f}")
    
    # Print group summary
    print(f"\n  Group importance summary:")
    for _, row in group_importance_df.iterrows():
        if row['total_shap_importance'] > 0:
            print(f"    {row['group']}: {row['normalized_importance']:.1f}% ({row['n_features']} features)")
    
    return group_importance_df, feature_importance_df

# ======================================================
# 7. VISUALIZATION FUNCTIONS (IMPROVED)
# ======================================================

def setup_plotting_style():
    """Set up publication-quality plotting style."""
    plt.style.use('seaborn-v0_8-whitegrid')
    mpl.rcParams['figure.figsize'] = [10, 6]
    mpl.rcParams['figure.dpi'] = 300
    mpl.rcParams['savefig.dpi'] = 300
    mpl.rcParams['font.size'] = 11
    mpl.rcParams['axes.titlesize'] = 14
    mpl.rcParams['axes.labelsize'] = 12
    mpl.rcParams['xtick.labelsize'] = 10
    mpl.rcParams['ytick.labelsize'] = 10
    mpl.rcParams['legend.fontsize'] = 10
    mpl.rcParams['figure.titlesize'] = 16

def plot_shap_summary(shap_values, feature_names, X_data, track, output_dir, top_n=20):
    """
    Create SHAP summary plot (beeswarm plot).
    """
    print(f" Creating SHAP summary plot for {track}...")
    
    # Limit to top N features for clarity
    if len(feature_names) > top_n:
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        top_indices = np.argsort(mean_abs_shap)[-top_n:][::-1]
        
        shap_values_top = shap_values[:, top_indices]
        feature_names_top = [feature_names[i] for i in top_indices]
        
        if isinstance(X_data, pd.DataFrame):
            X_data_top = X_data.iloc[:, top_indices]
        else:
            X_data_top = X_data[:, top_indices]
    else:
        shap_values_top = shap_values
        feature_names_top = feature_names
        X_data_top = X_data
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create SHAP summary plot
    shap.summary_plot(
        shap_values_top,
        X_data_top,
        feature_names=feature_names_top,
        show=False,
        max_display=top_n,
        plot_size=None,
        color_bar_label='Feature value',
        alpha=0.7
    )
    
    # Customize plot
    plt.title(f'SHAP Summary Plot - {track.upper()}\n(Top {len(feature_names_top)} Features)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('SHAP value (impact on model output)', fontsize=12)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, f'shap_summary_{track}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"   SHAP summary plot saved to: {plot_path}")
    
    # Also create a bar plot version
    plot_shap_bar(shap_values, feature_names, track, output_dir, top_n)

def plot_shap_bar(shap_values, feature_names, track, output_dir, top_n=20):
    """
    Create SHAP bar plot (feature importance).
    """
    # Calculate mean absolute SHAP
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Get top N features
    top_indices = np.argsort(mean_abs_shap)[-top_n:][::-1]
    top_features = [feature_names[i] for i in top_indices]
    top_shap = mean_abs_shap[top_indices]
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create horizontal bar plot
    y_pos = np.arange(len(top_features))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
    bars = ax.barh(y_pos, top_shap, color=colors)
    
    # Customize plot
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Mean |SHAP value| (average impact magnitude)', fontsize=12)
    ax.set_title(f'Feature Importance (SHAP) - {track.upper()}\n(Top {top_n} Features)', 
                 fontsize=16, fontweight='bold', pad=20)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, top_shap)):
        width = bar.get_width()
        ax.text(width * 1.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
                va='center', fontsize=9)
    
    # Add grid
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, f'shap_bar_{track}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"   SHAP bar plot saved to: {plot_path}")

def plot_group_importance(group_importance_df, track, output_dir):
    """
    Create group-level importance bar plot.
    """
    print(f" Creating group-level importance plot for {track}...")
    
    # Filter out groups with zero importance
    plot_df = group_importance_df[group_importance_df['total_shap_importance'] > 0].copy()
    
    if plot_df.empty:
        print("   No groups with importance > 0 to plot")
        return
    
    # Sort by importance
    plot_df = plot_df.sort_values('total_shap_importance', ascending=True)
    
    # Create color palette
    colors = plt.cm.Set3(np.linspace(0, 1, len(plot_df)))
    
    # Create plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Plot 1: Total SHAP importance
    bars1 = ax1.barh(plot_df['group'], plot_df['total_shap_importance'], color=colors)
    ax1.set_xlabel('Total SHAP Importance (Σ|SHAP|)', fontsize=12)
    ax1.set_title(f'Group-Level SHAP Importance - {track.upper()}\n(Sum of Absolute SHAP Values)', 
                  fontsize=14, fontweight='bold', pad=20)
    
    # Add value labels
    for bar in bars1:
        width = bar.get_width()
        ax1.text(width * 1.01, bar.get_y() + bar.get_height()/2, f'{width:.3f}', 
                va='center', fontsize=10)
    
    # Plot 2: Normalized importance
    bars2 = ax2.barh(plot_df['group'], plot_df['normalized_importance'], color=colors)
    ax2.set_xlabel('Normalized Importance (%)', fontsize=12)
    ax2.set_title(f'Group-Level Relative Importance - {track.upper()}\n(Percentage of Total)', 
                  fontsize=14, fontweight='bold', pad=20)
    
    # Add value labels
    for bar in bars2:
        width = bar.get_width()
        ax2.text(width * 1.01, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', 
                va='center', fontsize=10)
    
    # Add grid to both plots
    ax1.grid(True, alpha=0.3, axis='x')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Add number of features as text in bars
    for i, (bar1, bar2, row) in enumerate(zip(bars1, bars2, plot_df.itertuples())):
        # Add feature count to first bar
        ax1.text(bar1.get_width() * 0.1, bar1.get_y() + bar1.get_height()/2, 
                f'n={row.n_features}', va='center', fontsize=9, color='white', fontweight='bold')
    
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, f'group_importance_{track}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"   Group importance plot saved to: {plot_path}")

# ======================================================
# 8. MAIN ANALYSIS FUNCTION FOR EACH TRACK
# ======================================================

def analyze_track_complete(track_name, config):
    """
    Complete SHAP analysis for a single track.
    
    Parameters:
    -----------
    track_name : str
        'track1' or 'track2'
    config : dict
        Configuration dictionary
    
    Returns:
    --------
    dict : Analysis results
    """
    print(f"\n{'='*80}")
    print(f" STARTING COMPLETE SHAP ANALYSIS FOR {track_name.upper()}")
    print(f"{'='*80}")
    
    # Set paths based on track
    if track_name == 'track1':
        model_path = config['track1_model_path']
        features_path = config['track1_features_path']
        output_dir = config['track1_output_dir']
    elif track_name == 'track2':
        model_path = config['track2_model_path']
        features_path = config['track2_features_path']
        output_dir = config['track2_output_dir']
    else:
        raise ValueError("track_name must be 'track1' or 'track2'")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load model and feature information
    model_info = load_model_and_features(model_path, features_path, track_name)
    if model_info is None:
        print(f" Failed to load model for {track_name}")
        return None
    
    # 2. Prepare data with feature alignment
    data_dict = prepare_data_with_feature_alignment(
        config['train_path'],
        config['test_path'],
        model_info.get('feature_names', []),
        track_name
    )
    
    # 3. Compute SHAP values with safety checks
    explainer, shap_values, X_sample = compute_shap_values_safely(
        model_info['model'],
        data_dict['X_test'],
        data_dict['feature_names'],
        n_samples=config['n_samples_for_shap']
    )
    
    # 4. Aggregate by groups
    group_importance_df, feature_importance_df = aggregate_shap_by_groups(
        shap_values, 
        data_dict['feature_names'], 
        data_dict['feature_groups'], 
        track_name
    )
    
    # 5. Save results
    print("\n Saving analysis results...")
    
    # Save SHAP values
    shap_values_path = os.path.join(output_dir, 'shap_values.npy')
    np.save(shap_values_path, shap_values)
    print(f"  SHAP values saved to: {shap_values_path}")
    
    # Save group importance
    group_csv_path = os.path.join(output_dir, 'group_importance.csv')
    # Don't save the 'features' column as it contains lists
    group_to_save = group_importance_df.drop('features', axis=1) if 'features' in group_importance_df.columns else group_importance_df
    group_to_save.to_csv(group_csv_path, index=False, float_format='%.6f')
    print(f"   Group importance saved to: {group_csv_path}")
    
    # Save feature importance
    feature_csv_path = os.path.join(output_dir, 'feature_importance.csv')
    feature_importance_df.to_csv(feature_csv_path, index=False, float_format='%.6f')
    print(f"   Feature importance saved to: {feature_csv_path}")
    
    # Save metadata
    metadata_path = os.path.join(output_dir, 'analysis_metadata.json')
    with open(metadata_path, 'w') as f:
        metadata = {
            'track': track_name,
            'model_path': model_path,
            'n_features_analyzed': len(data_dict['feature_names']),
            'n_samples_shap': shap_values.shape[0],
            'expected_value': float(explainer.expected_value),
            'shap_values_shape': list(shap_values.shape),
            'feature_groups': {k: len(v) for k, v in data_dict['feature_groups'].items()},
            'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        json.dump(metadata, f, indent=4)
    print(f"   Metadata saved to: {metadata_path}")
    
    # 6. Create visualizations
    print("\n Creating visualizations...")
    setup_plotting_style()
    
    # SHAP summary plot
    plot_shap_summary(
        shap_values, data_dict['feature_names'], X_sample,
        track_name, output_dir, top_n=min(config['top_n_features'], len(data_dict['feature_names']))
    )
    
    # Group importance plot
    plot_group_importance(group_importance_df, track_name, output_dir)
    
    # 7. Print summary
    print(f"\n COMPLETE SUMMARY FOR {track_name.upper()}:")
    print(f"  Model: {model_info.get('experiment_name', 'Unknown experiment')}")
    print(f"  Features analyzed: {len(data_dict['feature_names'])}")
    print(f"  Samples used for SHAP: {shap_values.shape[0]}")
    print(f"  Feature groups: {len(group_importance_df)}")
    
    if len(group_importance_df) > 0:
        top_group = group_importance_df.iloc[0]
        print(f"  Most important group: {top_group['group']} ({top_group['normalized_importance']:.1f}%)")
    
    if len(feature_importance_df) > 0:
        top_feature = feature_importance_df.iloc[0]
        print(f"  Most important feature: {top_feature['feature']} (SHAP = {top_feature['mean_abs_shap']:.4f})")
    
    print(f"  Results saved in: {output_dir}")
    
    return {
        'model': model_info['model'],
        'explainer': explainer,
        'shap_values': shap_values,
        'X_sample': X_sample,
        'group_importance': group_importance_df,
        'feature_importance': feature_importance_df,
        'feature_names': data_dict['feature_names'],
        'feature_groups': data_dict['feature_groups'],
        'track': track_name,
        'metadata': metadata
    }

# ======================================================
# 9. COMPARISON AND REPORT GENERATION
# ======================================================

def create_comparison_report(track1_results, track2_results, output_dir):
    """
    Create comparison report between Track 1 and Track 2.
    """
    print(f"\n{'='*80}")
    print(f" CREATING COMPARISON REPORT")
    print(f"{'='*80}")
    
    if track1_results is None or track2_results is None:
        print(" Cannot create comparison report: missing results")
        return
    
    # Create comparison DataFrame
    comparison_data = []
    
    # Track 1 groups
    for _, row in track1_results['group_importance'].iterrows():
        if row['total_shap_importance'] > 0:
            comparison_data.append({
                'track': 'Track 1 (Dynamic only)',
                'group': row['group'],
                'importance': row['normalized_importance'],
                'n_features': row['n_features']
            })
    
    # Track 2 groups (excluding Gs if it's 0)
    for _, row in track2_results['group_importance'].iterrows():
        if row['total_shap_importance'] > 0:
            comparison_data.append({
                'track': 'Track 2 (Dynamic + Static)',
                'group': row['group'],
                'importance': row['normalized_importance'],
                'n_features': row['n_features']
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Save comparison data
    comparison_path = os.path.join(output_dir, 'track_comparison.csv')
    comparison_df.to_csv(comparison_path, index=False, float_format='%.2f')
    print(f" Comparison data saved to: {comparison_path}")
    
    # Create comparison plot
    create_comparison_plot(comparison_df, output_dir)
    
    # Generate detailed report
    generate_detailed_report(track1_results, track2_results, output_dir)

def create_comparison_plot(comparison_df, output_dir):
    """
    Create comparison plot between tracks.
    """
    # Pivot for plotting
    pivot_df = comparison_df.pivot_table(index='group', columns='track', 
                                        values='importance', fill_value=0)
    
    # Sort by Track 2 importance
    if 'Track 2 (Dynamic + Static)' in pivot_df.columns:
        pivot_df = pivot_df.sort_values('Track 2 (Dynamic + Static)', ascending=False)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Plot settings
    bar_width = 0.35
    x = np.arange(len(pivot_df))
    
    # Create bars
    if 'Track 1 (Dynamic only)' in pivot_df.columns:
        bars1 = ax.bar(x - bar_width/2, pivot_df['Track 1 (Dynamic only)'], 
                       bar_width, label='Track 1 (Dynamic only)', alpha=0.8, color='steelblue')
    
    if 'Track 2 (Dynamic + Static)' in pivot_df.columns:
        bars2 = ax.bar(x + bar_width/2, pivot_df['Track 2 (Dynamic + Static)'], 
                       bar_width, label='Track 2 (Dynamic + Static)', alpha=0.8, color='darkorange')
    
    # Customize plot
    ax.set_xlabel('Feature Group', fontsize=12)
    ax.set_ylabel('Normalized Importance (%)', fontsize=12)
    ax.set_title('Feature Group Importance Comparison\nTrack 1 vs Track 2', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot_df.index, rotation=45, ha='right', fontsize=10)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bars in [bars1, bars2] if 'bars1' in locals() and 'bars2' in locals() else []:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                       f'{height:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, 'track_comparison.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f" Comparison plot saved to: {plot_path}")

def generate_detailed_report(track1_results, track2_results, output_dir):
    """
    Generate a detailed text report.
    """
    report_path = os.path.join(output_dir, 'shap_analysis_report.txt')
    
    with open(report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("SHAP ANALYSIS REPORT - TRACK 1 & TRACK 2\n")
        f.write("="*80 + "\n\n")
        
        f.write("1. TRACK 1 (Dynamic features only)\n")
        f.write("-"*40 + "\n")
        f.write(f"Features analyzed: {len(track1_results['feature_names'])}\n")
        f.write(f"Samples used for SHAP: {track1_results['shap_values'].shape[0]}\n")
        f.write(f"Expected value (base): {track1_results['explainer'].expected_value:.4f}\n\n")
        
        f.write("Group importance:\n")
        for _, row in track1_results['group_importance'].iterrows():
            if row['total_shap_importance'] > 0:
                f.write(f"  {row['group']}: {row['normalized_importance']:.1f}% ({row['n_features']} features)\n")
        
        f.write("\nTop 5 features:\n")
        for i, (_, row) in enumerate(track1_results['feature_importance'].head(5).iterrows()):
            f.write(f"  {i+1}. {row['feature']}: {row['mean_abs_shap']:.4f}\n")
        
        f.write("\n\n2. TRACK 2 (Dynamic + Static features)\n")
        f.write("-"*40 + "\n")
        f.write(f"Features analyzed: {len(track2_results['feature_names'])}\n")
        f.write(f"Samples used for SHAP: {track2_results['shap_values'].shape[0]}\n")
        f.write(f"Expected value (base): {track2_results['explainer'].expected_value:.4f}\n\n")
        
        f.write("Group importance:\n")
        for _, row in track2_results['group_importance'].iterrows():
            if row['total_shap_importance'] > 0:
                f.write(f"  {row['group']}: {row['normalized_importance']:.1f}% ({row['n_features']} features)\n")
        
        f.write("\nTop 5 features:\n")
        for i, (_, row) in enumerate(track2_results['feature_importance'].head(5).iterrows()):
            f.write(f"  {i+1}. {row['feature']}: {row['mean_abs_shap']:.4f}\n")
        
        f.write("\n\n3. KEY INSIGHTS\n")
        f.write("-"*40 + "\n")
        
        # Static feature analysis
        if 'Gs' in track2_results['group_importance']['group'].values:
            static_row = track2_results['group_importance'][track2_results['group_importance']['group'] == 'Gs'].iloc[0]
            if static_row['total_shap_importance'] > 0:
                f.write(f"• Static features (Gs) contribute {static_row['normalized_importance']:.1f}% to model predictions\n")
                f.write(f"  Number of static features: {static_row['n_features']}\n")
        
        # Comparison insights
        f.write("\n• Feature count comparison:\n")
        f.write(f"  Track 1: {len(track1_results['feature_names'])} features\n")
        f.write(f"  Track 2: {len(track2_results['feature_names'])} features\n")
        f.write(f"  Additional features in Track 2: {len(track2_results['feature_names']) - len(track1_results['feature_names'])}\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f" Detailed report saved to: {report_path}")

# ======================================================
# 10. MAIN EXECUTION FUNCTION
# ======================================================

def main_complete():
    """
    Complete main function to run SHAP analysis for both tracks.
    """
    print("="*80)
    print(" COMPLETE XGBOOST SHAP ANALYSIS - TRACK 1 & TRACK 2")
    print("="*80)
    print("This script will:")
    print("  1. Load saved XGBoost models for both tracks")
    print("  2. Align features between models and data")
    print("  3. Compute SHAP values for test data")
    print("  4. Aggregate importance by feature groups")
    print("  5. Generate publication-ready visualizations")
    print("  6. Create comparison reports")
    print("="*80)
    
    # Create output directories
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    os.makedirs(CONFIG['track1_output_dir'], exist_ok=True)
    os.makedirs(CONFIG['track2_output_dir'], exist_ok=True)
    
    # Check if files exist
    print("\n Checking required files...")
    
    required_files = [
        ('Track 1 model', CONFIG['track1_model_path']),
        ('Track 1 features', CONFIG['track1_features_path']),
        ('Track 2 model', CONFIG['track2_model_path']),
        ('Track 2 features', CONFIG['track2_features_path']),
        ('Training data', CONFIG['train_path']),
        ('Test data', CONFIG['test_path'])
    ]
    
    all_files_exist = True
    for name, path in required_files:
        if os.path.exists(path):
            print(f"   {name}: Found")
        else:
            print(f"   {name}: Not found at {path}")
            all_files_exist = False
    
    if not all_files_exist:
        print("\n Some required files are missing. Please check the paths.")
        return
    
    # Run analysis for Track 1
    print("\n" + "="*80)
    print("ANALYZING TRACK 1...")
    print("="*80)
    
    track1_results = analyze_track_complete('track1', CONFIG)
    
    if track1_results is None:
        print(" Failed to analyze Track 1")
        return
    
    # Run analysis for Track 2
    print("\n" + "="*80)
    print("ANALYZING TRACK 2...")
    print("="*80)
    
    track2_results = analyze_track_complete('track2', CONFIG)
    
    if track2_results is None:
        print(" Failed to analyze Track 2")
        # Continue with just Track 1 results
        print("\n Continuing with Track 1 results only...")
        track2_results = None
    
    # Create comparison report if both tracks were successful
    if track1_results is not None and track2_results is not None:
        create_comparison_report(track1_results, track2_results, CONFIG['output_dir'])
    
    # Final summary
    print("\n" + "="*80)
    print(" ANALYSIS COMPLETE!")
    print("="*80)
    
    print(f"\n RESULTS SAVED IN:")
    print(f"  Track 1: {CONFIG['track1_output_dir']}")
    print(f"  Track 2: {CONFIG['track2_output_dir']}")
    print(f"  Comparisons: {CONFIG['output_dir']}")
    
    print(f"\n FILES GENERATED FOR EACH TRACK:")
    print("  1. shap_values.npy - Raw SHAP values")
    print("  2. group_importance.csv - Group-level importance")
    print("  3. feature_importance.csv - Feature-level importance")
    print("  4. shap_summary_[track].png - SHAP beeswarm plot")
    print("  5. shap_bar_[track].png - Feature importance bar plot")
    print("  6. group_importance_[track].png - Group importance plot")
    print("  7. analysis_metadata.json - Analysis metadata")
    
    if track1_results is not None and track2_results is not None:
        print(f"\n COMPARISON FILES:")
        print("  1. track_comparison.csv - Numerical comparison")
        print("  2. track_comparison.png - Visual comparison")
        print("  3. shap_analysis_report.txt - Detailed report")
    


# ======================================================
# 11. QUICK ANALYSIS FUNCTIONS
# ======================================================

def quick_track_analysis(track_name):
    """
    Quick analysis for a single track.
    """
    if track_name not in ['track1', 'track2']:
        print(" Invalid track name. Use 'track1' or 'track2'.")
        return
    
    print(f"\n QUICK ANALYSIS FOR {track_name.upper()}")
    print("="*60)
    
    results = analyze_track_complete(track_name, CONFIG)
    
    if results is not None:
        print(f"\n QUICK SUMMARY FOR {track_name.upper()}:")
        print(f"  Features: {len(results['feature_names'])}")
        print(f"  Samples used: {results['shap_values'].shape[0]}")
        
        print(f"\n  Top 5 features by SHAP importance:")
        for i, (_, row) in enumerate(results['feature_importance'].head(5).iterrows()):
            print(f"    {i+1}. {row['feature']}: {row['mean_abs_shap']:.4f}")
        
        print(f"\n  Top 3 groups by SHAP importance:")
        for i, (_, row) in enumerate(results['group_importance'].head(3).iterrows()):
            if row['total_shap_importance'] > 0:
                print(f"    {row['group']}: {row['normalized_importance']:.1f}%")
    
    return results

# ======================================================
# 12. DIRECT EXECUTION
# ======================================================

if __name__ == "__main__":
    """
    Entry point for running the complete SHAP analysis.
    """
    print("="*80)
    print("XGBOOST SHAP ANALYSIS - COMPLETE SOLUTION")
    print("="*80)
    print("\nOptions:")
    print("  1. Run complete analysis for both tracks (main_complete())")
    print("  2. Quick analysis for Track 1 (quick_track_analysis('track1'))")
    print("  3. Quick analysis for Track 2 (quick_track_analysis('track2'))")
    print("\nRunning complete analysis...")
    print("="*80)
    
    try:
        main_complete()
    except Exception as e:
        print(f"\n Error in main analysis: {e}")
        import traceback
        traceback.print_exc()
        
        print(f"\n Trying individual track analysis...")
        print(f"\n{'='*60}")
        print("ANALYZING TRACK 1...")
        print("="*60)
        try:
            track1_results = quick_track_analysis('track1')
        except Exception as e1:
            print(f" Failed Track 1: {e1}")
        
        print(f"\n{'='*60}")
        print("ANALYZING TRACK 2...")
        print("="*60)
        try:
            track2_results = quick_track_analysis('track2')
        except Exception as e2:
            print(f" Failed Track 2: {e2}")
        
        print(f"\n{'='*80}")
        print("ANALYSIS ATTEMPTED - CHECK OUTPUT DIRECTORIES")
        print("="*80)